In [2]:
import pandas as pd
from db_queries import get_location_metadata
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import numpy as np
from matplotlib.ticker import MaxNLocator
from pathlib import Path
import yaml
import xarray as xr

# Add FHS forecasted q0s to plots
import xarray as xr
import os

# Set paths ####################################################################
RAW_DATA_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/training_data/2026_10_08.01/data.parquet"
GBD_FILE = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/gbd_prevalence/gbd_mean_neonatal_mortality_prevalence.parquet"
FHS_PATH1 = "/mnt/share/forecasting/data/32/future/death/20260716_q0_reference_s130v98_nq0"  # Reference Q0 without ODA effects, pre-shifting to 2025 retro
FHS_PATH2 = "/mnt/share/forecasting/data/32/future/death/20260717_q0_reference_s130v108_noreagg_nq0"  # Reference q0 with ODA effects, pre-shifting to 2025 retro

FORMATTED_DIR = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/fhs_q0s"

# INFERENCE_FILE_09_14 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_14.01/ssp245.parquet"  # Consumption merged on birth year
# RESULTS_09_14 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_14.02/"  # .01 is the input spec; .02 is the fit

# RESULTS_09_21 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_21.02/"
# RESULTS_09_22 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_22.02/"
# RESULTS_09_23 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_23.02/"  # birth_year dropped, consumption_median_RT rescaled to $/day
# RESULTS_09_24 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_23.03/"  # birth_year dropped, consumption_median_RT, climate variables included
# RESULTS_09_25 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_25.01/"  # final specification, sex_id modified
# RESULTS_09_28 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/models/2026_09_25.01/"  # final specification, sex_id modified


# INFERENCE_FILE_09_21 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_22.01/ssp245.parquet"  # Inclusion of consumption_pd_national_median (logged)
# INFERENCE_FILE_09_22 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_22.02/ssp245.parquet"  # Inclusion of consumption_pd_national_median (logged)
# INFERENCE_FILE_09_23 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_23.01/ssp245.parquet"  # model 2026_09_23.02: consumption_median_RT (LDI based), no birth_year
# INFERENCE_FILE_09_24 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_23.02/ssp245.parquet"  # model 2026_09_24: birth_year dropped, consumption_median_RT, climate variables included
# INFERENCE_FILE_09_25 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_25.02/ssp245.parquet"  # final specification, sex_id modified
# INFERENCE_FILE_09_28 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_09_28.01/ssp245.parquet"  # final specification, sex_id modified
INFERENCE_FILE_10_08 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_10_08.01/ssp245.parquet"


PLOT_OUTPUT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/"


df_raw = pd.read_parquet(RAW_DATA_PATH)

# Add location names
locs = get_location_metadata(location_set_id=39, release_id=16)

df_raw = df_raw.merge(
    locs[["location_id", "location_name", "level", "super_region_name"]],
    on="location_id",
    how="left",
)

df_raw.head()

,index,year_start,year_end,nid,survey_name,int_year,int_month,sex_id,mothers_age_year,aod_months,...,q99_prev_6_mo_avg,q99_prev_9_mo_avg,zone,year_id,age_group_id,int_birth_year_diff_months,consumption_median_RT,location_name,level,super_region_name
0,10,2009,2010,13109,DHS_MIS,None,None,2,None,0,...,0.000000,0.000000,23.0,2009,42,0,2.361256,Uganda,3,Sub-Saharan Africa
1,11,2022,2022,538795,None,2022,5,2,18,0,...,0.833333,0.555556,24.0,2021,42,11,3.404259,United Republic of Tanzania,3,Sub-Saharan Africa
2,16,2017,2017,353526,None,2017,4,1,20,0,...,1.833333,1.222222,27.0,2017,42,0,4.210878,Senegal,3,Sub-Saharan Africa
3,24,2019,2021,467681,DHS,None,None,1,None,0,...,0.000000,0.000000,24.0,2020,42,2,5.111802,India,3,South Asia
4,25,2000,2000,19156,None,2000,2,2,24,0,...,0.000000,0.000000,25.0,1999,42,2,2.106530,Cambodia,3,"Southeast Asia, East Asia, and Oceania"


In [3]:
year5q0_no_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH1, "year5q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)
year5q0_with_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH2, "year5q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)
day28q0_no_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH1, "day28q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)
day28q0_with_cuts = (
    (
        xr.open_dataarray(os.path.join(FHS_PATH2, "day28q0.nc"))
        .sel(scenario=130, sex_id=[1, 2])
        .mean(dim="draw")
        .squeeze()
    )
    .to_dataframe()
    .reset_index()
)

In [4]:
# Two subsets of the residual step's prevalence plots ########################
# Reproduces what make_results_plots does, in this kernel, so it needs no
# rra_climate_health import:
#   1. read sev_means.parquet, clip the shifted columns to [0, 1] as the
#      diagnostics do
#   2. population-weight the most-detailed locations up the FHS hierarchy
#   3. draw one page per location, same line styles as the published pdf
# Then write two pdfs: Global + the 7 super regions, and the level-3 countries.
SUB_RV = "2026_10_08.01"
SUB_MEASURE = "neonatal_mortality"
SUB_ROOT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition"
SUB_SEV = f"{SUB_ROOT}/{SUB_MEASURE}/results/{SUB_RV}/sev_means.parquet"
SUB_HIER = f"{SUB_ROOT}/input/fhs_hierarchy.parquet"
SUB_POP_FUT = "/mnt/share/forecasting/data/32/future/population/future_population_s130v41/population.nc"
SUB_POP_PAST = "/mnt/share/forecasting/data/16/past/population/20250603_etl_run_id_417/population.nc"
SUB_POP_SCENARIO = 130
SUB_REFERENCE = "ssp245"
SUB_ANCHOR = 2023  # constants.LAST_GBD_YEAR, where the shift is read off

# line styles copied from residual_diagnostics.PREVALENCE_STYLE_MAP / LABEL_MAP
SUB_STYLE = [
    ("gbd_value", "black", "-", "GBD Prevalence"),
    ("adjusted_constant_climate", "lightgreen", "-", "Shifted constant climate"),
    ("adjusted_ssp585", "purple", "-", "Shifted SSP 5-8.5"),
    ("adjusted_ssp126", "green", "-", "Shifted SSP 1-2.6"),
    ("model_value", "blue", ":", "Original model"),
    ("residual_model_value", "blue", "--", "Residual model"),
    ("adjusted_model_value", "blue", "-", "Shifted model"),
]

sub_res = pd.read_parquet(SUB_SEV)
sub_hier = pd.read_parquet(SUB_HIER)
_ages = sorted(sub_res.index.get_level_values("age_group_id").unique())
_sexes = sorted(sub_res.index.get_level_values("sex_id").unique())
_years = sorted(sub_res.index.get_level_values("year_id").unique())

# the clipping make_results_plots applies before aggregating
for _c in [c for c in sub_res.columns if c.startswith("adjusted_")]:
    sub_res[_c] = sub_res[_c].clip(0, 1)


# Reported age groups that the population files store only as their parts,
# mirroring constants.AGE_GROUP_AGGREGATES. Neonatal's 42 is early + late
# neonatal, so the population has to be summed over [2, 3] first.
SUB_AGE_AGGREGATES = {
    4: [388, 389],
    5: [238, 34],
    42: [2, 3],
    1: [2, 3, 388, 389, 238, 34],
}


def sub_load_population():
    """Past + future population means, on the sev_means demographic grid."""
    detailed = sorted({a for age in _ages for a in SUB_AGE_AGGREGATES.get(age, [age])})
    fut = (
        xr.open_dataset(SUB_POP_FUT)
        .sel(age_group_id=detailed, sex_id=_sexes, scenario=SUB_POP_SCENARIO)
        .to_dataframe()
        .reset_index()
    )
    past = (
        xr.open_dataset(SUB_POP_PAST)
        .sel(age_group_id=detailed, sex_id=_sexes)
        .to_dataframe()
        .reset_index()
    )
    out = []
    for df in (past, fut):
        val = "population" if "population" in df.columns else "draws"
        keys = ["location_id", "year_id", "age_group_id", "sex_id"]
        df = (
            df.groupby(keys, as_index=False)[val]
            .mean()
            .rename(columns={val: "population"})
        )
        # roll the detailed groups back up into the reported ones
        parts = []
        for age in _ages:
            members = SUB_AGE_AGGREGATES.get(age, [age])
            sub = df[df.age_group_id.isin(members)].copy()
            sub["age_group_id"] = age
            parts.append(sub.groupby(keys, as_index=False)["population"].sum())
        out.append(pd.concat(parts))
    pop = pd.concat(out)
    # past and future overlap; the forecast file wins, matching the pipeline
    pop = pop.drop_duplicates(
        ["location_id", "year_id", "age_group_id", "sex_id"], keep="last"
    )
    return pop[pop.year_id.isin(_years)]


sub_pop = sub_load_population()

# FHS forecast q0s, for reference against the pipeline's own lines. These files
# already carry every level of the hierarchy - Global, super regions, regions,
# countries and subnationals - at age_group_id 42, so they merge straight on
# without any aggregation of their own. Values are plain probabilities
# (rate_denominator = 1), the same scale as gbd_value, so they only need x100.
SUB_FHS = {
    "fhs_with_cuts": (
        "/mnt/share/forecasting/data/32/future/death/"
        "20260717_q0_reference_s130v108_noreagg_nq0",
        "#d1495b",
        "-",
        "FHS forecasts with ODA cuts",
    ),
    "fhs_no_cuts": (
        "/mnt/share/forecasting/data/32/future/death/"
        "20260716_q0_reference_s130v98_nq0",
        "#d1495b",
        "--",
        "FHS forecasts without ODA cuts",
    ),
}


def sub_load_fhs():
    """Draw-averaged day28q0 for both ODA scenarios, keyed like sub_agg."""
    out = None
    for col, (path, *_rest) in SUB_FHS.items():
        da = xr.open_dataarray(f"{path}/day28q0.nc")
        df = (
            da.sel(scenario=130, sex_id=_sexes)
            .mean(dim="draw")
            .squeeze()
            .to_dataframe()
            .reset_index()
            .rename(columns={"value": col})
        )
        keys = ["location_id", "year_id", "sex_id", "age_group_id"]
        df = df[keys + [col]]
        out = df if out is None else out.merge(df, on=keys, how="outer")
    return out


sub_fhs = sub_load_fhs()
print(
    f"FHS overlay: {sub_fhs.location_id.nunique()} locations, "
    f"{int(sub_fhs.year_id.min())}-{int(sub_fhs.year_id.max())}, "
    f"age {sorted(sub_fhs.age_group_id.unique())}"
)

# population-weight every most-detailed location up its path_to_top_parent
_detailed = sub_hier[sub_hier["most_detailed"] == 1][
    ["location_id", "path_to_top_parent"]
]
_vals = [c for c, *_ in SUB_STYLE if c in sub_res.columns]
_flat = (
    sub_res.reset_index()
    .merge(_detailed, on="location_id", how="inner")
    .merge(
        sub_pop, on=["location_id", "year_id", "age_group_id", "sex_id"], how="inner"
    )
)
_flat["agg_id"] = _flat["path_to_top_parent"].str.split(",")
_exp = _flat.explode("agg_id")
_exp["agg_id"] = _exp["agg_id"].astype(int)
for _c in _vals:
    _exp[_c] = _exp[_c] * _exp["population"]
_keys = ["agg_id", "year_id", "age_group_id", "sex_id"]
sub_agg = _exp.groupby(_keys, as_index=False)[_vals + ["population"]].sum(min_count=1)
for _c in _vals:
    sub_agg[_c] = sub_agg[_c] / sub_agg["population"]
sub_agg = sub_agg.rename(columns={"agg_id": "location_id"})

# Raw survey data, to overlay on the country pages. It is coded to country
# level and by BIRTH year, plotted against the model's year axis; sex_id is a
# string in the training extract so it needs coercing to match.
SUB_MIN_BIRTHS = 100
sub_obs = (
    df_raw.assign(sex_id=lambda d: pd.to_numeric(d["sex_id"], errors="coerce"))
    .dropna(subset=["sex_id", "birth_year", "neonatal_mortality"])
    .groupby(["location_id", "birth_year", "sex_id"], observed=True)
    .agg(rate=("neonatal_mortality", "mean"), n=("neonatal_mortality", "size"))
    .reset_index()
    .query("n >= @SUB_MIN_BIRTHS")
)
sub_obs["sex_id"] = sub_obs["sex_id"].astype(int)
print(
    f"survey overlay: {len(sub_obs):,} country/year/sex cells with "
    f">= {SUB_MIN_BIRTHS} births, {sub_obs.location_id.nunique()} countries"
)

_lv = sub_hier.set_index("location_id")["level"]
_nm = sub_hier.set_index("location_id")["location_name"]
print(
    f"aggregated to {sub_agg.location_id.nunique()} locations "
    f"from {_flat.location_id.nunique()} most-detailed inputs; "
    f"levels {sorted(set(_lv.reindex(sub_agg.location_id.unique()).dropna().astype(int)))}"
)


def sub_make_pdf(loc_ids, out_path, title_note, *, overlay_survey=False):
    """One page per location: a sex-by-age grid, styled like the published pdf."""
    with PdfPages(out_path) as pdf:
        for loc_id in loc_ids:
            fig, axes = plt.subplots(
                len(_sexes),
                len(_ages),
                figsize=(14, 8),
                sharex=True,
                sharey=True,
                squeeze=False,
            )
            handles, labels = [], []
            for j, sex_id in enumerate(_sexes):
                for i, age_id in enumerate(_ages):
                    ax = axes[j, i]
                    blk = sub_agg[
                        (sub_agg.location_id == loc_id)
                        & (sub_agg.sex_id == sex_id)
                        & (sub_agg.age_group_id == age_id)
                    ].sort_values("year_id")
                    if overlay_survey:
                        # behind the lines; point size scales with births so
                        # thin survey years read as less reliable
                        o = sub_obs[
                            (sub_obs.location_id == loc_id) & (sub_obs.sex_id == sex_id)
                        ]
                        if len(o):
                            ax.scatter(
                                o.birth_year,
                                o.rate,
                                s=np.clip(o.n / 60, 4, 60),
                                color="0.45",
                                alpha=0.55,
                                linewidths=0,
                                zorder=0,
                                label=f"survey data (size = births, "
                                f"min {SUB_MIN_BIRTHS})",
                            )
                    for col, colour, ls, lab in SUB_STYLE:
                        if col in blk.columns:
                            ax.plot(
                                blk.year_id, blk[col], color=colour, ls=ls, label=lab
                            )
                    # FHS reference forecasts; 2024-2050 only, so these appear
                    # as a segment rather than spanning the panel
                    f = sub_fhs[
                        (sub_fhs.location_id == loc_id)
                        & (sub_fhs.sex_id == sex_id)
                        & (sub_fhs.age_group_id == age_id)
                    ].sort_values("year_id")
                    for col, (_pth, colour, ls, lab) in SUB_FHS.items():
                        if col in f.columns and f[col].notna().any():
                            ax.plot(
                                f.year_id,
                                f[col],
                                color=colour,
                                ls=ls,
                                lw=1.8,
                                zorder=5,
                                label=lab,
                            )
                    # the anchor year: the shifted model meets GBD here by
                    # construction, and GBD itself stops here
                    ax.axvline(
                        SUB_ANCHOR,
                        color="0.45",
                        lw=1.1,
                        ls="-.",
                        zorder=1,
                        label=f"anchor year {SUB_ANCHOR}",
                    )
                    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
                    if j == 0:
                        ax.set_title(f"Age group {age_id}")
                    ax.set_ylabel("Male" if sex_id == 1 else "Female")
                    if i > 0:
                        ax.set_ylabel("")
                        ax.tick_params(labelleft=False)
                    if j == len(_sexes) - 1:
                        ax.set_xlabel("Year")
                    if not handles:
                        handles, labels = ax.get_legend_handles_labels()
            fig.suptitle(
                f"{SUB_MEASURE.capitalize()} Prevalence - Location {loc_id}: "
                f"{_nm.get(loc_id)}",
                fontsize=16,
            )
            fig.legend(
                handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0)
            )
            fig.tight_layout(rect=[0, 0.05, 1, 1])
            pdf.savefig(fig)
            plt.close(fig)
    print(f"Saved {out_path} ({len(loc_ids)} pages, {title_note})")


_ordered = [l for l in sub_agg.location_id.unique() if l in _lv.index]
# The survey data is coded to country level, so it goes on the country pages
# only; putting a national rate on a super-region panel would mix grains.
SUB_SETS = {
    "global_superregions": (
        [l for l in _ordered if _lv[l] in (0, 1)],
        "Global and the super regions",
        False,
    ),
    "countries": ([l for l in _ordered if _lv[l] == 3], "level-3 countries", True),
}
for _tag, (_locs, _note, _ov) in SUB_SETS.items():
    sub_make_pdf(
        _locs,
        f"{PLOT_OUTPUT}{SUB_MEASURE}_prevalence_plots_{_tag}.pdf",
        _note,
        overlay_survey=_ov,
    )

FHS overlay: 538 locations, 2024-2050, age [42]
survey overlay: 1,711 country/year/sex cells with >= 100 births, 60 countries
aggregated to 513 locations from 473 most-detailed inputs; levels [0, 1, 2, 3, 4]
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/neonatal_mortality_prevalence_plots_global_superregions.pdf (8 pages, Global and the super regions)
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/neonatal_mortality_prevalence_plots_countries.pdf (204 pages, level-3 countries)


In [5]:
# Interactive explorer for the residual-step prevalence results ##############
# Writes one self-contained HTML file to PLOT_OUTPUT: the same series the pdfs
# draw, but searchable by location and inspectable point by point. No CDN, no
# build step - the data is embedded and the chart is hand-rolled SVG, so it
# opens off the shared drive with nothing else available.
#
# Depends on sub_agg / sub_fhs / sub_hier from the cell above.
import html
import json

ART_PATH = f"{PLOT_OUTPUT}{SUB_MEASURE}_prevalence_explorer.html"
ART_ANCHOR = SUB_ANCHOR

# Series encoding. Colours are the dataviz reference palette, validated with
# the all-pairs gate in BOTH modes (worst CVD dE 9.2 light / 9.4 dark, worst
# normal-vision dE 24.0 / 20.9). Only THREE categorical hues are used: four or
# more cannot clear the all-pairs floors, and every line here shares one panel,
# so every pair really is on screen together.
#   blue   = our model, its three stages separated by dash pattern, not hue
#   orange = SSP 5-8.5 (high emissions), aqua = SSP 1-2.6 (low emissions)
#   ink / muted grey = external references (GBD, FHS); achromatic on purpose,
#                      so they read as context rather than competing for a hue
# Trailing int is paint order, low to high: the shifted model is the headline
# series, so it is drawn last and sits on top of everything it crosses.
ART_SERIES = [
    ("gbd_value", "GBD prevalence", "ink", "none", 2.5, "Reference", 5),
    ("model_value", "Original model", "blue", "2 3", 2, "Model", 1),
    ("residual_model_value", "Residual model", "blue", "7 4", 2, "Model", 2),
    ("adjusted_model_value", "Shifted model", "blue", "none", 2.5, "Model", 9),
    ("adjusted_ssp126", "Shifted SSP 1-2.6", "aqua", "none", 2, "Scenario", 4),
    ("adjusted_ssp585", "Shifted SSP 5-8.5", "orange", "none", 2, "Scenario", 3),
    ("fhs_no_cuts", "FHS, no ODA cuts", "muted", "7 4", 2, "Reference", 6),
    ("fhs_with_cuts", "FHS, with ODA cuts", "muted", "none", 2, "Reference", 7),
]

# ---- national median consumption, the dual-axis context line ----------------
# The RT income series the model's consumption_median_RT predictor comes from:
# LDI_VERSION national distributions at the reference scenario and the 50th
# population percentile, divided to dollars per day.
#
# It is published for COUNTRIES only (204 of them), so it is pushed down to each
# most-detailed location via that location's level-3 ancestor - a subnational's
# "national median" is its country's by definition - and then population-weighted
# back up the hierarchy with the same weights the prevalence series use. For a
# super region the result is a population-weighted mean of country medians, not a
# true regional median; it is a context line, not an estimate.
ART_INCOME_VERSION = "v9"
ART_INCOME_LABEL = "National median consumption"

_inc_nat = pd.read_parquet(
    f"{SUB_ROOT}/input/ldi/{ART_INCOME_VERSION}/national_estimates.parquet",
    columns=["location_id", "year_id", "scenario", "population_percentile", "ldipc"],
)
_inc_nat = _inc_nat[
    (_inc_nat.scenario == "reference") & (_inc_nat.population_percentile == 0.5)
][["location_id", "year_id", "ldipc"]]
_inc_nat["income"] = _inc_nat["ldipc"] / 365

# level-3 ancestor of every most-detailed location
_p2t = sub_hier.set_index("location_id")["path_to_top_parent"]
_lvl = sub_hier.set_index("location_id")["level"]


def _country_of(loc_id):
    for _a in (int(x) for x in _p2t.get(loc_id, "").split(",") if x):
        if _lvl.get(_a) == 3:
            return _a
    return None


_det_ids = sorted(sub_res.index.get_level_values("location_id").unique())
_ctry = pd.DataFrame(
    {"location_id": _det_ids, "country_id": [_country_of(i) for i in _det_ids]}
).dropna()
_ctry["country_id"] = _ctry["country_id"].astype(int)

# total population per most-detailed location/year, summed over age and sex
_pop_ly = sub_pop.groupby(["location_id", "year_id"], as_index=False)[
    "population"
].sum()
_inc_det = (
    _ctry.merge(
        _inc_nat[["location_id", "year_id", "income"]],
        left_on="country_id",
        right_on="location_id",
        how="inner",
        suffixes=("", "_c"),
    )
    .drop(columns=["location_id_c"])
    .merge(_pop_ly, on=["location_id", "year_id"], how="inner")
    .merge(
        sub_hier[["location_id", "path_to_top_parent"]], on="location_id", how="inner"
    )
)
_inc_det["agg_id"] = _inc_det["path_to_top_parent"].str.split(",")
_inc_exp = _inc_det.explode("agg_id")
_inc_exp["agg_id"] = _inc_exp["agg_id"].astype(int)
_inc_exp["w"] = _inc_exp["income"] * _inc_exp["population"]
_inc_agg = _inc_exp.groupby(["agg_id", "year_id"], as_index=False)[
    ["w", "population"]
].sum(min_count=1)
_inc_agg["income"] = _inc_agg["w"] / _inc_agg["population"]
_inc_agg = _inc_agg.rename(columns={"agg_id": "location_id"})[
    ["location_id", "year_id", "income"]
]
print(
    f"income overlay: ldi {ART_INCOME_VERSION}, {_inc_agg.location_id.nunique()} "
    f"locations, {int(_inc_agg.year_id.min())}-{int(_inc_agg.year_id.max())}, "
    f"${_inc_agg.income.min():.2f}-${_inc_agg.income.max():.2f}/day"
)

# ---- assemble the payload ---------------------------------------------------
_art_years = sorted(sub_agg.year_id.unique())
_art_yidx = {y: i for i, y in enumerate(_art_years)}
_art_cols = [c for c, *_ in ART_SERIES if c in sub_agg.columns]
_art_fhs_cols = [c for c in ("fhs_no_cuts", "fhs_with_cuts") if c in sub_fhs.columns]

# Sexes stay separate. A combined line is deliberately not offered: the FHS
# series carry no population, so theirs could only be an unweighted mean of two
# differently sized groups, which would not be comparable with the
# population-weighted model lines beside it.
_art_long = sub_agg
_art_fhs = sub_fhs

_art_meta = sub_hier.set_index("location_id")
_LV_NAME = {
    0: "Global",
    1: "Super region",
    2: "Region",
    3: "Country",
    4: "Subnational",
    5: "Subnational",
}


def _art_round(v):
    """1e-6 of a probability - 10x finer than the 3-decimal percent on display."""
    return None if v is None or pd.isna(v) else round(float(v), 6)


_art_locs = {}
for _lid, _g in _art_long.groupby("location_id"):
    if _lid not in _art_meta.index:
        continue
    _row = _art_meta.loc[_lid]
    _by_sex = {}
    for _sex, _gs in _g.groupby("sex_id"):
        _gs = _gs.set_index("year_id")
        _f = _art_fhs[(_art_fhs.location_id == _lid) & (_art_fhs.sex_id == _sex)]
        _f = _f.set_index("year_id")
        _ser = {}
        for _c in _art_cols:
            _v = _gs[_c].reindex(_art_years)
            _ser[_c] = [_art_round(x) for x in _v]
        for _c in _art_fhs_cols:
            _v = _f[_c].reindex(_art_years) if len(_f) else None
            _ser[_c] = (
                [None] * len(_art_years) if _v is None else [_art_round(x) for x in _v]
            )
        _by_sex[str(int(_sex))] = _ser
    _inc_l = _inc_agg[_inc_agg.location_id == _lid].set_index("year_id")["income"]
    _inc_l = _inc_l.reindex(_art_years)
    _art_locs[str(int(_lid))] = {
        "inc": [None if pd.isna(x) else round(float(x), 4) for x in _inc_l],
        "n": str(_row["location_name"]),
        "lv": int(_row["level"]),
        "lvn": _LV_NAME.get(int(_row["level"]), "Other"),
        "sr": (
            None
            if pd.isna(_row.get("super_region_name"))
            else str(_row.get("super_region_name"))
        ),
        "s": _by_sex,
    }

# ---- summary statistics shown under the chart -------------------------------
# Country-level, population-weighted across sexes, so these are one number per
# country rather than one per sex. Cumulative differences are summed over years
# and carry units of percentage-point-years (pp-yr): a country 1 pp above GBD
# for 10 years scores 10.
_sum_keys = ["location_id", "year_id"]
_sw = sub_agg.copy()
_sum_cols = [
    c
    for c in ("gbd_value", "adjusted_model_value", "adjusted_ssp126", "adjusted_ssp585")
    if c in _sw.columns
]
for _c in _sum_cols:
    _sw[_c] = _sw[_c] * _sw["population"]
_sum = _sw.groupby(_sum_keys, as_index=False)[_sum_cols + ["population"]].sum(
    min_count=1
)
for _c in _sum_cols:
    _sum[_c] = _sum[_c] / _sum["population"]
_sum["level"] = _sum.location_id.map(sub_hier.set_index("location_id")["level"])
_sum = _sum[_sum.level == 3]

# 1. distance from GBD, over the years GBD actually covers
_g = _sum[_sum.gbd_value.notna()].copy()
_g["d"] = 100 * (_g.adjusted_model_value - _g.gbd_value)
_gbd_rank = (
    _g.groupby("location_id")
    .agg(cum=("d", "sum"), n=("d", "size"))
    .assign(mean=lambda d: d.cum / d.n)
    .reindex(columns=["cum", "mean", "n"])
    .sort_values("cum", key=lambda c: c.abs(), ascending=False)
    .head(3)
)
_gbd_years = (int(_g.year_id.min()), int(_g.year_id.max()))

# 2. spread between the two climate scenarios, over the full horizon
_s = _sum.copy()
_s["d"] = 100 * (_s.adjusted_ssp585 - _s.adjusted_ssp126)
_scen_rank = (
    _s.groupby("location_id")
    .agg(cum=("d", "sum"), n=("d", "size"))
    .assign(mean=lambda d: d.cum / d.n)
    .sort_values("cum", key=lambda c: c.abs(), ascending=False)
    .head(3)
)
_scen_years = (int(_s.year_id.min()), int(_s.year_id.max()))

# 3. the residual model's fitted SDI slope per super region.
# Not stored anywhere by the pipeline, so it is recovered from the output: the
# model is intercept(location_age_sex) + slope(super_region) * sdi, so within one
# location/age/sex an OLS of the predicted residual on SDI returns that region's
# slope EXACTLY - provided the SDI series is the one the model was fitted with.
# rra-climate-health_2 was repointed from the round-32 SDI release to round-38 on
# 2026-10-07, so which applies depends on when the residual step ran. Rather than
# hard-code a guess, every candidate is tried and the one that reproduces a
# straight line wins: the wrong vintage leaves visible curvature (~1e-4) where
# the right one lands at machine precision (~1e-7).
ART_SDI_VINTAGES = {
    "round-38 s159v5 (scenario 159)": (
        "/mnt/share/forecasting/data/38/past/sdi/past_sdi_s159v5/sdi.nc",
        "/mnt/share/forecasting/data/38/future/sdi/future_sdi_s159v5/sdi.nc",
        159,
    ),
    "round-32 s130v89 (scenario 130)": (
        "/mnt/share/forecasting/data/32/past/sdi/past_sdi_s130v89/sdi.nc",
        "/mnt/share/forecasting/data/32/future/sdi/future_sdi_s130v89/sdi.nc",
        130,
    ),
}


def _art_load_sdi(past_path, future_path, scenario):
    _p = xr.open_dataset(past_path).to_dataframe().reset_index()
    _p = _p.groupby(["location_id", "year_id"], as_index=False)["draws"].mean()
    _f = xr.open_dataset(future_path).to_dataframe().reset_index()
    _f = _f[_f.scenario == scenario]
    _f = _f.groupby(["location_id", "year_id"], as_index=False)["draws"].mean()
    return (
        pd.concat([_p, _f])
        .rename(columns={"draws": "sdi"})
        .drop_duplicates(["location_id", "year_id"])
    )


def _art_recover_slopes(sdi_df):
    """Per-location OLS of the predicted residual on SDI, and how straight it was."""
    _r = sub_res.reset_index()
    _r["pr"] = _r.residual_model_value - _r.model_value
    _r = _r.merge(sdi_df, on=["location_id", "year_id"], how="inner")
    _r["srn"] = _r.location_id.map(
        sub_hier.set_index("location_id")["super_region_name"]
    )
    rows, worst = [], 0.0
    for (_lid, _a, _x), _gg in _r.groupby(["location_id", "age_group_id", "sex_id"]):
        if _gg.sdi.nunique() < 3 or pd.isna(_gg.srn.iloc[0]):
            continue
        _sl, _ic = np.polyfit(_gg.sdi, _gg.pr, 1)
        worst = max(worst, float(np.abs(_gg.pr - (_sl * _gg.sdi + _ic)).max()))
        rows.append((_gg.srn.iloc[0], float(_sl)))
    return pd.DataFrame(rows, columns=["srn", "slope"]), worst


_sdi_fits = {}
for _name, (_pp, _ff, _sc) in ART_SDI_VINTAGES.items():
    try:
        _sdi_fits[_name] = _art_recover_slopes(_art_load_sdi(_pp, _ff, _sc))
    except (FileNotFoundError, OSError) as _err:
        print(f"  SDI vintage {_name} unavailable: {_err}")
if not _sdi_fits:
    msg = "No SDI vintage could be read; cannot recover the residual slopes."
    raise RuntimeError(msg)
ART_SDI_USED = min(_sdi_fits, key=lambda k: _sdi_fits[k][1])
_slope_df, _slope_dev = _sdi_fits[ART_SDI_USED]
print(
    "SDI vintage matched to this residual: "
    + ART_SDI_USED
    + "  ("
    + ", ".join(f"{k.split()[0]} dev {v[1]:.1e}" for k, v in _sdi_fits.items())
    + ")"
)
_slopes = (
    _slope_df.groupby("srn")["slope"]
    .agg(["median", "std", "size"])
    .sort_values("median")
)
print(
    f"summary: GBD window {_gbd_years[0]}-{_gbd_years[1]}, "
    f"scenario window {_scen_years[0]}-{_scen_years[1]}, "
    f"{len(_slopes)} super-region slopes "
    f"(max within-region spread {_slopes['std'].max():.1e})"
)
print(_slopes.to_string(float_format=lambda v: f"{v:10.5f}"))

# GBD vs the FHS no-ODA-cuts forecast. FHS carries no population of its own, so
# it is collapsed across sexes with the model's population weights for the same
# location/year/sex. NOTE the two series overlap in ONE year only: GBD ends at
# 2024 and FHS begins at 2024, so this is that year's gap, not a sum over time.
_fh = sub_fhs.merge(
    sub_agg[["location_id", "year_id", "sex_id", "age_group_id", "population"]],
    on=["location_id", "year_id", "sex_id", "age_group_id"],
    how="inner",
)
_fh["w"] = _fh["fhs_no_cuts"] * _fh["population"]
_fh = _fh.groupby(["location_id", "year_id"], as_index=False)[["w", "population"]].sum(
    min_count=1
)
_fh["fhs_no_cuts"] = _fh["w"] / _fh["population"]
_gf = _sum[_sum.gbd_value.notna()].merge(
    _fh[["location_id", "year_id", "fhs_no_cuts"]],
    on=["location_id", "year_id"],
    how="inner",
)
_gf["d"] = 100 * (_gf.gbd_value - _gf.fhs_no_cuts)
_fhs_rank = (
    _gf.groupby("location_id")
    .agg(cum=("d", "sum"), n=("d", "size"))
    .assign(mean=lambda d: d.cum / d.n)
    .sort_values("cum", key=lambda c: c.abs(), ascending=False)
    .head(3)
)
_fhs_years = sorted(_gf.year_id.unique())
print(
    f"GBD vs FHS overlap: {len(_fhs_years)} year(s) "
    f"({_fhs_years[0]}-{_fhs_years[-1]}) across {_gf.location_id.nunique()} countries"
)

_nm_of = sub_hier.set_index("location_id")["location_name"]
ART_SUMMARY = {
    "gbdWindow": list(_gbd_years),
    "scenWindow": list(_scen_years),
    "gbd": [
        {
            "id": int(i),
            "n": str(_nm_of.get(i)),
            "cum": round(float(r.cum), 2),
            "mean": round(float(r["mean"]), 3),
        }
        for i, r in _gbd_rank.iterrows()
    ],
    "fhsWindow": [int(_fhs_years[0]), int(_fhs_years[-1])],
    "fhs": [
        {
            "id": int(i),
            "n": str(_nm_of.get(i)),
            "cum": round(float(r.cum), 2),
            "mean": round(float(r["mean"]), 3),
        }
        for i, r in _fhs_rank.iterrows()
    ],
    "scen": [
        {
            "id": int(i),
            "n": str(_nm_of.get(i)),
            "cum": round(float(r.cum), 2),
            "mean": round(float(r["mean"]), 3),
        }
        for i, r in _scen_rank.iterrows()
    ],
    "sdiVintage": ART_SDI_USED,
    "slopes": [
        {
            "sr": str(i),
            "slope": round(float(r["median"]), 4),
            "sd": round(float(r["std"]), 5),
        }
        for i, r in _slopes.iterrows()
    ],
}

ART_PAYLOAD = {
    "measure": SUB_MEASURE.replace("_", " "),
    "resultsVersion": SUB_RV,
    "anchor": ART_ANCHOR,
    "years": [int(y) for y in _art_years],
    "income": {"label": ART_INCOME_LABEL, "version": ART_INCOME_VERSION},
    "series": [
        {"k": k, "l": lab, "c": col, "d": dash, "w": w, "g": grp, "z": z}
        for k, lab, col, dash, w, grp, z in ART_SERIES
        if k in _art_cols + _art_fhs_cols
    ],
    "locs": _art_locs,
    "summary": ART_SUMMARY,
}
print(
    f"explorer payload: {len(ART_PAYLOAD['locs'])} locations, "
    f"{len(ART_PAYLOAD['years'])} years, {len(ART_PAYLOAD['series'])} series"
)
ART_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<style>
:root{
  color-scheme:light;
  --plane:#f9f9f7; --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781;
  --grid:#e1e0d9; --axis:#c3c2b7; --ring:rgba(11,11,11,.10);
  --blue:#2a78d6; --orange:#eb6834; --aqua:#1baf7a; --neutral:#898781; --inkline:#0b0b0b;
  --accent:#2a78d6;
}
@media (prefers-color-scheme:dark){ :root:where(:not([data-theme="light"])){
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --blue:#3987e5; --orange:#d95926; --aqua:#199e70; --neutral:#898781; --inkline:#fff;
  --accent:#3987e5;
}}
:root[data-theme="dark"]{
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --blue:#3987e5; --orange:#d95926; --aqua:#199e70; --neutral:#898781; --inkline:#fff;
  --accent:#3987e5;
}
*{box-sizing:border-box}
body{margin:0;background:var(--plane);color:var(--ink);
  font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
header{display:flex;align-items:baseline;gap:12px;flex-wrap:wrap;
  padding:14px 16px;border-bottom:1px solid var(--ring)}
h1{font-size:17px;margin:0;font-weight:650}
.sub{color:var(--ink2);font-size:12.5px}
.spacer{flex:1}
button,select,input{font:inherit;color:var(--ink);background:var(--surface);
  border:1px solid var(--ring);border-radius:7px;padding:6px 10px}
button{cursor:pointer}
button:hover{border-color:var(--muted)}
button[aria-pressed="true"]{background:var(--accent);border-color:var(--accent);color:#fff}
.wrap{display:grid;grid-template-columns:286px minmax(0,1fr);gap:16px;padding:16px;
  align-items:start}
.panel{background:var(--surface);border:1px solid var(--ring);border-radius:11px}
.wrap2{padding:0 16px 16px}
#side{padding:12px;position:sticky;top:16px;max-height:calc(100vh - 32px);
  display:flex;flex-direction:column}
#q{width:100%;margin-bottom:8px}
#list{flex:1;min-height:0;overflow:auto;margin:0;padding:0;list-style:none}
#list li{padding:6px 9px;border-radius:7px;cursor:pointer;display:flex;gap:8px;
  align-items:baseline}
#list li:hover{background:var(--plane)}
#list li[aria-selected="true"]{background:var(--accent);color:#fff}
#list li[aria-selected="true"] .lv{color:#fff;opacity:.85}
.lv{color:var(--muted);font-size:11px;margin-left:auto;white-space:nowrap}
.grp{padding:9px 9px 3px;font-size:11px;letter-spacing:.06em;text-transform:uppercase;
  color:var(--muted);font-weight:650}
#main{padding:14px 16px 18px}
.row{display:flex;gap:16px;align-items:center;flex-wrap:wrap;margin-bottom:10px}
.ctl{display:flex;gap:6px;align-items:center}
.ctl label{color:var(--ink2);font-size:12.5px}
h2{font-size:16px;margin:0 0 2px}
.meta{color:var(--ink2);font-size:12.5px;margin-bottom:10px}
#legend{display:flex;flex-wrap:wrap;gap:6px 14px;margin:10px 0 2px}
.lg{display:flex;gap:7px;align-items:center;cursor:pointer;border:0;background:none;
  padding:3px 5px;border-radius:6px;color:var(--ink2);font-size:12.5px}
.lg:hover{background:var(--plane)}
.lg[aria-pressed="false"]{opacity:.38}
.lg svg{flex:none}
.grpname{font-size:11px;color:var(--muted);align-self:center;margin-right:-6px}
figure{margin:0}
#chart{width:100%;height:auto;display:block;touch-action:none}
#tip{position:fixed;pointer-events:none;z-index:9;background:var(--surface);
  border:1px solid var(--ring);border-radius:9px;padding:9px 11px;font-size:12.5px;
  box-shadow:0 6px 22px rgba(0,0,0,.14);opacity:0;transition:opacity .08s;max-width:290px}
#tip b{font-weight:650}
#tip table{border-collapse:collapse;margin-top:5px}
#tip td{padding:1px 0}
#tip td.v{text-align:right;padding-left:12px;font-variant-numeric:tabular-nums}
#tbl{width:100%;border-collapse:collapse;font-size:12.5px;
  font-variant-numeric:tabular-nums}
#tbl th,#tbl td{padding:4px 8px;text-align:right;border-bottom:1px solid var(--grid)}
#tbl th:first-child,#tbl td:first-child{text-align:left}
#tbl thead th{position:sticky;top:0;background:var(--surface);color:var(--ink2);
  font-weight:600}
#tblwrap{max-height:62vh;overflow:auto;margin-top:10px}
.hidden{display:none !important}
#sumgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(290px,1fr));gap:18px}
.sumcard h3{font-size:13px;margin:0 0 2px;font-weight:650}
.sumcard p{margin:0 0 7px;color:var(--muted);font-size:11.5px}
.sumtbl{width:100%;border-collapse:collapse;font-size:12.5px;
  font-variant-numeric:tabular-nums}
.sumtbl th,.sumtbl td{padding:4px 7px;text-align:right;border-bottom:1px solid var(--grid)}
.sumtbl th:first-child,.sumtbl td:first-child{text-align:left}
.sumtbl thead th{color:var(--ink2);font-weight:600}
.sumtbl tbody tr{cursor:pointer}
.sumtbl tbody tr:hover{background:var(--plane)}
.sumtbl .neg{color:var(--ink2)}
footer{color:var(--muted);font-size:11.5px;padding:0 16px 22px;line-height:1.6}
kbd{border:1px solid var(--ring);border-bottom-width:2px;border-radius:4px;
  padding:0 4px;font:inherit;font-size:11px}
@media (max-width:860px){.wrap{grid-template-columns:1fr}#side{position:static}
  #list{max-height:230px}}
</style></head><body>
<header>
  <h1>__TITLE__</h1>
  <span class="sub">results __RV__ &middot; shifted to GBD at __ANCHOR__</span>
  <span class="spacer"></span>
  <button id="view" aria-pressed="false">Table view</button>
  <button id="csv">Download CSV</button>
  <button id="theme">Theme</button>
</header>
<div class="wrap">
  <nav class="panel" id="side">
    <input id="q" type="search" placeholder="Search locations&hellip;  ( / )"
           autocomplete="off" aria-label="Search locations">
    <ul id="list" role="listbox" aria-label="Locations"></ul>
  </nav>
  <section class="panel" id="main">
    <h2 id="locname">&nbsp;</h2>
    <div class="meta" id="locmeta">&nbsp;</div>
    <div class="row">
      <div class="ctl"><label for="sex">Sex</label>
        <select id="sex"><option value="1">Male</option>
          <option value="2">Female</option></select></div>
      <div class="ctl"><label for="scale">Y axis</label>
        <select id="scale"><option value="lin">Linear</option>
          <option value="log">Log</option></select></div>
      <div class="ctl"><label for="xmax">Through</label>
        <select id="xmax"><option value="2100">2100</option>
          <option value="2050">2050</option><option value="2035">2035</option></select></div>
    </div>
    <figure id="figwrap">
      <svg id="chart" role="img" aria-label="Prevalence over time"></svg>
      <div id="legend"></div>
    </figure>
    <div id="tblwrap" class="hidden"><table id="tbl"></table></div>
  </section>
</div>
<div class="wrap2">
  <section class="panel" id="summary">
    <h2>Summary statistics</h2>
    <div class="meta">Country level (level 3), population-weighted across sexes.
      Cumulative differences are summed over years, in percentage-point-years
      (pp-yr): 1 pp of separation sustained for 10 years scores 10.</div>
    <div id="sumgrid"></div>
  </section>
</div>
<div id="tip" role="tooltip"></div>
<footer>
  <div>Lines are the residual step's own output, population-weighted up the FHS
  hierarchy. The dashed vertical marks __ANCHOR__, the last GBD year and the point
  the shifted model is pinned to; GBD itself stops there. FHS reference forecasts
  cover 2024&ndash;2050 only.</div>
  <div>Sexes are shown separately: the FHS series carry no population, so a
  combined line could only be an unweighted mean and would not be comparable with
  the population-weighted model lines. On a log axis, values at or below zero
  cannot be drawn &mdash; the residual model runs negative in many locations, so
  use the linear axis to see it.</div>
  <div>The dashed grey line is national median consumption on the RIGHT axis, in
  dollars per day &mdash; a different quantity in different units, so the point
  where it crosses a prevalence line carries no meaning. It is published per
  country, so a subnational shows its country's value and an aggregate shows a
  population-weighted mean of country medians.</div>
  <div><kbd>/</kbd> search &middot; <kbd>&uarr;</kbd><kbd>&darr;</kbd> move
  &middot; <kbd>Enter</kbd> select &middot; click a legend entry to hide a series.</div>
</footer>
<script>
const D = __PAYLOAD__;
const S = D.series, YEARS = D.years;
const COLORVAR = {ink:'--inkline', blue:'--blue', orange:'--orange', aqua:'--aqua',
                  muted:'--neutral'};
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const ids = Object.keys(D.locs);
const LVORDER = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5};
ids.sort((a,b)=>{
  const A=D.locs[a], B=D.locs[b];
  return (LVORDER[A.lv]-LVORDER[B.lv]) || A.n.localeCompare(B.n);
});
let cur = ids.find(i=>D.locs[i].lv===0) || ids[0];
let hidden = new Set(), filtered = ids.slice(), hi = 0;

/* ---- location list ---- */
const listEl = document.getElementById('list');
function renderList(){
  const q = document.getElementById('q').value.trim().toLowerCase();
  filtered = ids.filter(i=>{
    const L = D.locs[i];
    return !q || L.n.toLowerCase().includes(q) ||
           (L.sr||'').toLowerCase().includes(q);
  });
  if(hi >= filtered.length) hi = Math.max(0, filtered.length-1);
  let out = '', last = null;
  filtered.forEach((i,k)=>{
    const L = D.locs[i];
    if(L.lvn !== last){ out += `<div class="grp">${L.lvn}</div>`; last = L.lvn; }
    out += `<li role="option" data-id="${i}" data-k="${k}"
      aria-selected="${i===cur}" tabindex="-1"><span>${esc(L.n)}</span>
      <span class="lv">${L.sr?esc(abbr(L.sr)):''}</span></li>`;
  });
  listEl.innerHTML = out || '<div class="grp">No match</div>';
}
const esc = s => s.replace(/[&<>"]/g, c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const abbr = s => s.replace('Sub-Saharan Africa','SSA')
  .replace('Central Europe, Eastern Europe, and Central Asia','C/E Europe & C Asia')
  .replace('Southeast Asia, East Asia, and Oceania','SE/E Asia & Oceania')
  .replace('Latin America and Caribbean','LatAm & Caribbean')
  .replace('North Africa and Middle East','N Africa & Middle East');

listEl.addEventListener('click', e=>{
  const li = e.target.closest('li'); if(!li) return;
  cur = li.dataset.id; hi = +li.dataset.k; location.hash = 'loc='+cur; draw();
});
const qEl = document.getElementById('q');
qEl.addEventListener('input', ()=>{ hi=0; renderList(); });
qEl.addEventListener('keydown', e=>{
  if(e.key==='ArrowDown'||e.key==='ArrowUp'){
    e.preventDefault();
    hi = Math.min(filtered.length-1, Math.max(0, hi + (e.key==='ArrowDown'?1:-1)));
    const li = listEl.querySelector(`li[data-k="${hi}"]`);
    if(li){ listEl.querySelectorAll('li').forEach(x=>x.setAttribute('aria-selected','false'));
            li.setAttribute('aria-selected','true'); li.scrollIntoView({block:'nearest'}); }
  } else if(e.key==='Enter' && filtered[hi]){
    cur = filtered[hi]; location.hash='loc='+cur; draw();
  }
});
document.addEventListener('keydown', e=>{
  if(e.key==='/' && document.activeElement!==qEl){ e.preventDefault(); qEl.focus(); qEl.select(); }
});

/* ---- chart ---- */
const svg = document.getElementById('chart'), tip = document.getElementById('tip');
const NS = 'http://www.w3.org/2000/svg';
const el = (t,a)=>{ const n=document.createElementNS(NS,t);
  for(const k in a) n.setAttribute(k,a[k]); return n; };
const fmt = v => v==null ? '—' : (100*v).toFixed(3)+'%';
let geom = null;

const INC_KEY='__income__';
const incOn = () => !!D.income && !hidden.has(INC_KEY);
const incData = () => (D.locs[cur].inc)||[];
const fmtMoney = v => v==null ? '\u2014' : '$'+v.toFixed(2);
function activeSeries(){ return S.filter(s=>!hidden.has(s.k)); }
// legend and tooltip keep the declared order; only painting is re-sorted
function paintOrder(){ return activeSeries().slice().sort((a,b)=>a.z-b.z); }
function seriesData(){
  const sex = document.getElementById('sex').value;
  return (D.locs[cur].s[sex]) || {};
}
function draw(){
  const L = D.locs[cur], data = seriesData();
  document.getElementById('locname').textContent = L.n;
  document.getElementById('locmeta').textContent =
    [L.lvn, L.sr||null, 'location_id '+cur].filter(Boolean).join('  ·  ');
  listEl.querySelectorAll('li').forEach(li=>
    li.setAttribute('aria-selected', String(li.dataset.id===cur)));
  renderLegend(); drawChart(data); renderTable(data);
}
function drawChart(data){
  const W = svg.clientWidth || 900, H = Math.max(330, Math.min(520, W*0.46));
  // the right gutter only opens up when the income axis is shown
  const m = {t:14, r: incOn()?62:16, b:34, l:58};
  svg.setAttribute('viewBox', `0 0 ${W} ${H}`);
  svg.setAttribute('height', H);
  svg.textContent = '';
  const xmax = +document.getElementById('xmax').value;
  const yrs = YEARS.filter(y=>y<=xmax);
  const logy = document.getElementById('scale').value === 'log';
  const act = activeSeries();
  let vals = [];
  act.forEach(s=>(data[s.k]||[]).forEach((v,i)=>{
    if(v!=null && YEARS[i]<=xmax && (!logy || v>0)) vals.push(v); }));
  if(!vals.length) vals = [0, 1e-3];
  // Linear: the domain is whatever the data spans, including negatives - the
  // residual model runs far below zero in many locations, and a floor of 0
  // would silently clip it. Log: non-positive values cannot be shown at all,
  // so they are dropped from `vals` above and the footnote says so.
  let lo, hiV = Math.max(...vals);
  if(logy){ lo = Math.min(...vals); if(hiV<=lo) hiV = lo*1.5+1e-9; lo*=0.85; hiV*=1.15; }
  else {
    lo = Math.min(0, Math.min(...vals));
    if(hiV<=lo) hiV = lo + 1e-9;
    const pad = (hiV-lo)*0.06; hiV += pad; if(lo<0) lo -= pad;
  }
  const x = y => m.l + (y-yrs[0])/(xmax-yrs[0] || 1)*(W-m.l-m.r);
  const Y = v => { if(v==null) return null;
    const f = logy ? (Math.log(Math.max(v,1e-12))-Math.log(lo))/(Math.log(hiV)-Math.log(lo))
                   : (v-lo)/(hiV-lo);
    return m.t + (1-f)*(H-m.t-m.b); };
  // Second scale for the income line. Deliberately NOT a peer of the
  // prevalence series: it is a different quantity in different units, so it
  // gets the neutral ink, its own right-hand axis, and axis labels in the same
  // colour so the pairing is explicit rather than inferred.
  let Y2 = null, iLo = 0, iHi = 1;
  if(incOn()){
    const iv = incData().filter((v,i)=>v!=null && YEARS[i]<=xmax);
    if(iv.length){
      iLo = Math.min(...iv); iHi = Math.max(...iv);
      if(iHi<=iLo) iHi = iLo + 1;
      const pad=(iHi-iLo)*0.08; iLo=Math.max(0,iLo-pad); iHi+=pad;
      Y2 = v => v==null ? null : m.t + (1-(v-iLo)/(iHi-iLo))*(H-m.t-m.b);
    }
  }
  geom = {x, Y, Y2, m, W, H, yrs, data, lo, hiV};

  const ticks = logy ? logTicks(lo,hiV) : linTicks(lo,hiV,5);
  ticks.forEach(t=>{
    const yy = Y(t); if(yy==null||yy<m.t-1||yy>H-m.b+1) return;
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:yy,y2:yy,stroke:css('--grid'),'stroke-width':1}));
    const lb = el('text',{x:m.l-8,y:yy+4,'text-anchor':'end','font-size':11,fill:css('--muted')});
    lb.textContent = (100*t).toFixed(t<0.01?2:(t<0.1?1:0))+'%'; svg.appendChild(lb);
  });
  const step = (xmax-yrs[0])>60?20:((xmax-yrs[0])>30?10:5);
  for(let yv=yrs[0]; yv<=xmax; yv+=step){
    const xx=x(yv);
    const lb = el('text',{x:xx,y:H-m.b+18,'text-anchor':'middle','font-size':11,fill:css('--muted')});
    lb.textContent = yv; svg.appendChild(lb);
  }
  svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:H-m.b,y2:H-m.b,
    stroke:css('--axis'),'stroke-width':1}));
  if(!logy && lo < 0){
    const z = Y(0);
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:z,y2:z,stroke:css('--axis'),
      'stroke-width':1.5}));
    const zl = el('text',{x:m.l+4,y:z-4,'font-size':11,fill:css('--muted')});
    zl.textContent='0'; svg.appendChild(zl);
  }
  if(D.anchor<=xmax){
    svg.appendChild(el('line',{x1:x(D.anchor),x2:x(D.anchor),y1:m.t,y2:H-m.b,
      stroke:css('--muted'),'stroke-width':1,'stroke-dasharray':'4 4'}));
    const a = el('text',{x:x(D.anchor)+5,y:m.t+12,'font-size':11,fill:css('--muted')});
    a.textContent = D.anchor; svg.appendChild(a);
  }
  paintOrder().forEach(s=>{
    const arr = data[s.k]||[]; let d='', pen=false;
    yrs.forEach((yv,i)=>{ const v=arr[i], yy=Y(v);
      if(v==null||yy==null||(logy&&v<=0)){ pen=false; return; }
      d += (pen?'L':'M') + x(yv).toFixed(1) + ' ' + yy.toFixed(1) + ' '; pen=true; });
    if(d) svg.appendChild(el('path',{d, fill:'none', stroke:css(COLORVAR[s.c]),
      'stroke-width':s.w, 'stroke-linecap':'round', 'stroke-linejoin':'round',
      ...(s.d!=='none'?{'stroke-dasharray':s.d}:{})}));
  });
  if(Y2){
    const ink2 = css('--ink2');
    linTicks(iLo,iHi,5).forEach(t=>{
      const yy=Y2(t); if(yy==null||yy<m.t-1||yy>H-m.b+1) return;
      const lb=el('text',{x:W-m.r+8,y:yy+4,'font-size':11,fill:ink2});
      lb.textContent='$'+t.toFixed(t<1?2:(t<10?1:0)); svg.appendChild(lb);
    });
    const cap=el('text',{x:W-m.r+8,y:m.t-2,'font-size':10,fill:ink2});
    cap.textContent='$/day'; svg.appendChild(cap);
    const arr=incData(); let d='', pen=false;
    yrs.forEach((yv,i)=>{ const v=arr[i], yy=Y2(v);
      if(v==null||yy==null){ pen=false; return; }
      d += (pen?'L':'M') + x(yv).toFixed(1) + ' ' + yy.toFixed(1) + ' '; pen=true; });
    if(d) svg.appendChild(el('path',{d, fill:'none', stroke:ink2,
      'stroke-width':1.8, 'stroke-dasharray':'6 3 1 3',
      'stroke-linecap':'round', 'stroke-linejoin':'round'}));
  }
  const cross = el('g',{id:'cross',visibility:'hidden'}); svg.appendChild(cross);
  cross.appendChild(el('line',{id:'cl',y1:m.t,y2:H-m.b,stroke:css('--muted'),'stroke-width':1}));
  svg.appendChild(el('rect',{x:m.l,y:m.t,width:Math.max(0,W-m.l-m.r),
    height:Math.max(0,H-m.t-m.b),fill:'transparent',id:'hit'}));
  const hit = svg.querySelector('#hit');
  hit.addEventListener('pointermove', onMove);
  hit.addEventListener('pointerleave', ()=>{ cross.setAttribute('visibility','hidden');
    tip.style.opacity=0; });
}
function linTicks(lo,hi,n){
  const raw=(hi-lo)/n, mag=Math.pow(10,Math.floor(Math.log10(raw)));
  const st=[1,2,2.5,5,10].map(m=>m*mag).find(s=>s>=raw)||mag*10;
  const out=[]; for(let v=Math.ceil(lo/st)*st; v<=hi; v+=st) out.push(v); return out;
}
function logTicks(lo,hi){
  const out=[]; for(let e=Math.floor(Math.log10(lo)); e<=Math.ceil(Math.log10(hi)); e++)
    for(const m of [1,2,5]){ const v=m*Math.pow(10,e); if(v>=lo&&v<=hi) out.push(v); }
  return out.length?out:[lo,hi];
}
function onMove(ev){
  if(!geom) return;
  const r = svg.getBoundingClientRect(), sc = geom.W / r.width;
  const px = (ev.clientX - r.left) * sc;
  let best=geom.yrs[0], bd=1e9;
  geom.yrs.forEach(y=>{ const d=Math.abs(geom.x(y)-px); if(d<bd){bd=d;best=y;} });
  const i = YEARS.indexOf(best);
  const cross = svg.querySelector('#cross'); cross.setAttribute('visibility','visible');
  const cl = cross.querySelector('#cl');
  cl.setAttribute('x1', geom.x(best)); cl.setAttribute('x2', geom.x(best));
  cross.querySelectorAll('circle').forEach(c=>c.remove());
  let rows='';
  activeSeries().forEach(s=>{
    const v=(geom.data[s.k]||[])[i]; if(v==null) return;
    const yy=geom.Y(v); if(yy==null) return;
    cross.appendChild(el('circle',{cx:geom.x(best),cy:yy,r:4,
      fill:css(COLORVAR[s.c]),stroke:css('--surface'),'stroke-width':2}));
    rows += `<tr><td><svg width="16" height="8"><line x1="0" y1="4" x2="16" y2="4"
      stroke="${css(COLORVAR[s.c])}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''}/></svg> ${esc(s.l)}</td>
      <td class="v">${fmt(v)}</td></tr>`;
  });
  if(incOn() && geom.Y2){
    const v=incData()[i];
    if(v!=null){
      const yy=geom.Y2(v), ink2=css('--ink2');
      cross.appendChild(el('circle',{cx:geom.x(best),cy:yy,r:4,fill:ink2,
        stroke:css('--surface'),'stroke-width':2}));
      rows += `<tr><td><svg width="16" height="8"><line x1="0" y1="4" x2="16" y2="4"
        stroke="${ink2}" stroke-width="1.8" stroke-dasharray="6 3 1 3"/></svg>
        ${esc(D.income.label)}</td><td class="v">${fmtMoney(v)}</td></tr>`;
    }
  }
  tip.innerHTML = `<b>${esc(D.locs[cur].n)} &middot; ${best}</b><table>${rows}</table>`;
  tip.style.opacity = 1;
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lft = ev.clientX + 16, top = ev.clientY - th/2;
  if(lft + tw > innerWidth - 8) lft = ev.clientX - tw - 16;
  tip.style.left = Math.max(8,lft)+'px';
  tip.style.top = Math.min(Math.max(8,top), innerHeight-th-8)+'px';
}
/* ---- legend ---- */
function renderLegend(){
  const box = document.getElementById('legend'); box.textContent='';
  let g = null;
  S.forEach(s=>{
    if(s.g!==g){ g=s.g; const t=document.createElement('span');
      t.className='grpname'; t.textContent=g; box.appendChild(t); }
    const b=document.createElement('button');
    b.className='lg'; b.setAttribute('aria-pressed', String(!hidden.has(s.k)));
    b.innerHTML = `<svg width="22" height="10"><line x1="0" y1="5" x2="22" y2="5"
      stroke="${css(COLORVAR[s.c])}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''} stroke-linecap="round"/></svg>
      <span>${esc(s.l)}</span>`;
    b.onclick = ()=>{ hidden.has(s.k)?hidden.delete(s.k):hidden.add(s.k); draw(); };
    box.appendChild(b);
  });
  if(D.income){
    const t=document.createElement('span');
    t.className='grpname'; t.textContent='Right axis'; box.appendChild(t);
    const b=document.createElement('button');
    b.className='lg'; b.setAttribute('aria-pressed', String(incOn()));
    b.innerHTML = `<svg width="22" height="10"><line x1="0" y1="5" x2="22" y2="5"
      stroke="${css('--ink2')}" stroke-width="1.8"
      stroke-dasharray="6 3 1 3" stroke-linecap="round"/></svg>
      <span>${esc(D.income.label)} ($/day)</span>`;
    b.onclick = ()=>{ hidden.has(INC_KEY)?hidden.delete(INC_KEY):hidden.add(INC_KEY); draw(); };
    box.appendChild(b);
  }
}
/* ---- table view ---- */
function renderTable(data){
  const act = activeSeries();
  let h = '<thead><tr><th>Year</th>' + act.map(s=>`<th>${esc(s.l)}</th>`).join('')
        + (incOn()?`<th>${esc(D.income.label)} ($/day)</th>`:'') + '</tr></thead><tbody>';
  const xmax = +document.getElementById('xmax').value;
  YEARS.forEach((y,i)=>{ if(y>xmax) return;
    h += `<tr><td>${y}</td>` + act.map(s=>`<td>${fmt((data[s.k]||[])[i])}</td>`).join('')
       + (incOn()?`<td>${fmtMoney(incData()[i])}</td>`:'') + '</tr>';
  });
  document.getElementById('tbl').innerHTML = h + '</tbody>';
}
document.getElementById('view').onclick = e=>{
  const showTable = e.target.getAttribute('aria-pressed') !== 'true';
  e.target.setAttribute('aria-pressed', String(showTable));
  document.getElementById('figwrap').classList.toggle('hidden', showTable);
  document.getElementById('tblwrap').classList.toggle('hidden', !showTable);
};
document.getElementById('csv').onclick = ()=>{
  const data = seriesData(), act = activeSeries();
  let csv = 'location_id,location_name,sex_id,year_id,' + act.map(s=>s.k).join(',')
    + (incOn()?',consumption_pd_RT':'') + '\n';
  const sex = document.getElementById('sex').value;
  YEARS.forEach((y,i)=>{ csv += `${cur},"${D.locs[cur].n}",${sex},${y},`
    + act.map(s=>{ const v=(data[s.k]||[])[i]; return v==null?'':v; }).join(',')
    + (incOn()?','+((incData()[i]==null)?'':incData()[i]):'') + '\n'; });
  const a=document.createElement('a');
  a.href = URL.createObjectURL(new Blob([csv],{type:'text/csv'}));
  a.download = `${D.measure.replace(/ /g,'_')}_${D.locs[cur].n.replace(/[^A-Za-z0-9]+/g,'_')}.csv`;
  a.click(); URL.revokeObjectURL(a.href);
};
document.getElementById('theme').onclick = ()=>{
  const now = document.documentElement.getAttribute('data-theme');
  const dark = now ? now==='dark'
    : matchMedia('(prefers-color-scheme: dark)').matches;
  document.documentElement.setAttribute('data-theme', dark?'light':'dark');
  draw(); renderSummary();
};
['sex','scale','xmax'].forEach(id=>document.getElementById(id).onchange = draw);
addEventListener('resize', ()=>{ if(geom) drawChart(seriesData()); });
/* ---- summary ---- */
function renderSummary(){
  const U = D.summary; if(!U) return;
  const rank = (rows, w) => `<table class="sumtbl"><thead><tr><th>Country</th>
    <th>Cumulative</th><th>Mean/yr</th></tr></thead><tbody>` + rows.map(r=>
    `<tr data-id="${r.id}" title="Show ${esc(r.n)}"><td>${esc(r.n)}</td>
     <td class="${r.cum<0?'neg':''}">${r.cum>0?'+':''}${r.cum.toFixed(1)} pp-yr</td>
     <td class="${r.mean<0?'neg':''}">${r.mean>0?'+':''}${r.mean.toFixed(2)} pp</td></tr>`
    ).join('') + '</tbody></table>';
  document.getElementById('sumgrid').innerHTML = `
    <div class="sumcard"><h3>Furthest from GBD</h3>
      <p>Shifted model minus GBD, ${U.gbdWindow[0]}&ndash;${U.gbdWindow[1]} (the
      years GBD covers). Ranked on absolute value.</p>${rank(U.gbd)}</div>
    <div class="sumcard"><h3>Widest scenario spread</h3>
      <p>Shifted SSP 5-8.5 minus Shifted SSP 1-2.6, ${U.scenWindow[0]}&ndash;${U.scenWindow[1]}.
      Ranked on absolute value.</p>${rank(U.scen)}</div>
    <div class="sumcard"><h3>Furthest from FHS</h3>
      <p>GBD minus the FHS no-ODA-cuts forecast. The two series meet in
      ${U.fhsWindow[0]===U.fhsWindow[1] ? U.fhsWindow[0]+' only - GBD ends there and FHS begins there'
        : U.fhsWindow[0]+'&ndash;'+U.fhsWindow[1]}, so this is that gap, not a
      sum over time. Ranked on absolute value.</p>${rank(U.fhs)}</div>
    <div class="sumcard"><h3>Residual model: SDI slope</h3>
      <p>Recovered against ${esc(U.sdiVintage||'the fitted SDI')}.</p>
      <table class="sumtbl"><thead><tr><th>Super region</th><th>Slope</th></tr></thead>
      <tbody>${U.slopes.map(r=>`<tr><td>${esc(r.sr)}</td>
        <td class="${r.slope<0?'neg':''}">${r.slope===0?'0 (clipped)':r.slope.toFixed(4)}</td>
        </tr>`).join('')}</tbody></table></div>`;
  document.querySelectorAll('#sumgrid tbody tr[data-id]').forEach(tr=>{
    tr.onclick = ()=>{ if(D.locs[tr.dataset.id]){ cur=tr.dataset.id;
      location.hash='loc='+cur; draw();
      document.getElementById('main').scrollIntoView({behavior:'smooth',block:'start'}); } };
  });
}
const hashId = (location.hash.match(/loc=(\d+)/)||[])[1];
if(hashId && D.locs[hashId]) cur = hashId;
renderList(); draw(); renderSummary();
</script></body></html>
"""

_art_title = f"{ART_PAYLOAD['measure'].title()} prevalence explorer"
_art_doc = (
    ART_HTML.replace("__TITLE__", html.escape(_art_title))
    .replace("__RV__", html.escape(SUB_RV))
    .replace("__ANCHOR__", str(ART_ANCHOR))
    # separators= drops the spaces json.dumps adds; </ is escaped so a value can
    # never close the <script> early
    .replace(
        "__PAYLOAD__",
        json.dumps(ART_PAYLOAD, separators=(",", ":")).replace("</", "<\\/"),
    )
)
Path(ART_PATH).write_text(_art_doc, encoding="utf-8")
print(f"Saved {ART_PATH} ({len(_art_doc) / 1e6:.1f} MB)")

income overlay: ldi v9, 513 locations, 2000-2100, $0.62-$320.07/day
SDI vintage matched to this residual: round-38 s159v5 (scenario 159)  (round-38 dev 1.0e-07, round-32 dev 2.6e-04)
summary: GBD window 2000-2024, scenario window 2000-2100, 7 super-region slopes (max within-region spread 2.1e-08)
                                                     median        std  size
srn                                                                         
North Africa and Middle East                       -0.04137    0.00000    42
Sub-Saharan Africa                                 -0.02838    0.00000   188
Latin America and Caribbean                        -0.01154    0.00000   180
High-income                                        -0.01009    0.00000   198
Southeast Asia, East Asia, and Oceania              0.00000    0.00000   198
South Asia                                          0.00000    0.00000    82
Central Europe, Eastern Europe, and Central Asia    0.00000    0.00000    58
GBD vs FH

In [ ]:
# Where neonatal mortality exceeds under-5 mortality ##########################
# Neonatal deaths are a SUBSET of under-5 deaths, so the neonatal line can never
# exceed the under-5 one. GBD never does: across 138,750 location-year-sex cells
# the neonatal share of under-5 peaks at 0.81 and is never >= 1. The two measures
# here are fitted as independent models with nothing tying them together, so the
# constraint is free to break - and it does.
#
# This cell aggregates both runs up the FHS hierarchy the same way the plots do,
# finds every location-year where the shifted neonatal model sits above the
# shifted under-5 model, and writes an interactive artifact of those locations.
import html
import json

XO_ROOT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition"
XO_NN = ("neonatal_mortality", "2026_10_08.01")  # this notebook's run
XO_CM = ("child_mortality", "2026_10_01.01")  # latest COMPLETE child run
XO_PATH = f"{PLOT_OUTPUT}neonatal_vs_under5_crossover.html"
XO_POP_FUT = "/mnt/share/forecasting/data/32/future/population/future_population_s130v41/population.nc"
XO_POP_PAST = "/mnt/share/forecasting/data/16/past/population/20250603_etl_run_id_417/population.nc"
XO_POP_SCENARIO = 130
XO_AGE_AGGREGATES = {
    4: [388, 389],
    5: [238, 34],
    42: [2, 3],
    1: [2, 3, 388, 389, 238, 34],
}
XO_VALS = ["adjusted_model_value", "model_value", "gbd_value"]

xo_hier = pd.read_parquet(f"{XO_ROOT}/input/fhs_hierarchy.parquet")
_xo_det = xo_hier[xo_hier["most_detailed"] == 1][["location_id", "path_to_top_parent"]]


def _xo_pop(ages, sexes, years):
    """Population on one measure's demographic grid, rolled up from its parts."""
    detailed = sorted({a for g in ages for a in XO_AGE_AGGREGATES.get(g, [g])})
    out = []
    for _path, _sel in ((XO_POP_PAST, {}), (XO_POP_FUT, {"scenario": XO_POP_SCENARIO})):
        _df = (
            xr.open_dataset(_path)
            .sel(age_group_id=detailed, sex_id=sexes, **_sel)
            .to_dataframe()
            .reset_index()
        )
        _v = "population" if "population" in _df.columns else "draws"
        _k = ["location_id", "year_id", "age_group_id", "sex_id"]
        _df = (
            _df.groupby(_k, as_index=False)[_v]
            .mean()
            .rename(columns={_v: "population"})
        )
        _parts = []
        for _g in ages:
            _s = _df[_df.age_group_id.isin(XO_AGE_AGGREGATES.get(_g, [_g]))].copy()
            _s["age_group_id"] = _g
            _parts.append(_s.groupby(_k, as_index=False)["population"].sum())
        out.append(pd.concat(_parts))
    _p = pd.concat(out).drop_duplicates(
        ["location_id", "year_id", "age_group_id", "sex_id"], keep="last"
    )
    return _p[_p.year_id.isin(years)]


def xo_aggregate(measure, rv):
    """Population-weight a results version up the hierarchy, as the plots do."""
    _s = pd.read_parquet(f"{XO_ROOT}/{measure}/results/{rv}/sev_means.parquet")
    for _c in [c for c in _s.columns if c.startswith("adjusted_")]:
        _s[_c] = _s[_c].clip(0, 1)
    _ages = sorted(_s.index.get_level_values("age_group_id").unique())
    _sexes = sorted(_s.index.get_level_values("sex_id").unique())
    _years = sorted(_s.index.get_level_values("year_id").unique())
    _vals = [c for c in XO_VALS if c in _s.columns]
    _f = (
        _s.reset_index()
        .merge(_xo_det, on="location_id", how="inner")
        .merge(
            _xo_pop(_ages, _sexes, _years),
            on=["location_id", "year_id", "age_group_id", "sex_id"],
            how="inner",
        )
    )
    _f["agg_id"] = _f["path_to_top_parent"].str.split(",")
    _e = _f.explode("agg_id")
    _e["agg_id"] = _e["agg_id"].astype(int)
    for _c in _vals:
        _e[_c] = _e[_c] * _e["population"]
    _a = _e.groupby(["agg_id", "year_id"], as_index=False)[_vals + ["population"]].sum(
        min_count=1
    )
    for _c in _vals:
        _a[_c] = _a[_c] / _a["population"]
    return _a.rename(columns={"agg_id": "location_id"})[
        ["location_id", "year_id"] + _vals
    ]


xo_nn = xo_aggregate(*XO_NN)
xo_cm = xo_aggregate(*XO_CM)
xo_m = xo_nn.merge(xo_cm, on=["location_id", "year_id"], suffixes=("_nn", "_cm"))
_xo_nm = xo_hier.set_index("location_id")["location_name"]
_xo_lv = xo_hier.set_index("location_id")["level"]
# Aggregates (Global, super regions, regions) are dropped: they are constructed
# by this very roll-up, so a crossover there is just the arithmetic consequence
# of the countries beneath it rather than an independent case. Only level 3 and
# below are places the models actually estimate.
XO_MIN_LEVEL = 3
xo_m = xo_m[xo_m.location_id.map(_xo_lv) >= XO_MIN_LEVEL]
xo_m["over"] = xo_m.adjusted_model_value_nn > xo_m.adjusted_model_value_cm
xo_bad = xo_m[xo_m.over]
_xo_sr = xo_hier.set_index("location_id")["super_region_name"]
_XO_LVN = {
    0: "Global",
    1: "Super region",
    2: "Region",
    3: "Country",
    4: "Subnational",
    5: "Subnational",
}
print(
    f"{len(xo_bad):,} location-years where neonatal > under-5, across "
    f"{xo_bad.location_id.nunique()} of {xo_m.location_id.nunique()} countries "
    f"and subnationals (aggregates excluded)"
)

# ---- observed survey rates from the training extracts ----------------------
# Both extracts are coded to COUNTRY level, so these points attach to level-3
# pages only; a subnational page shows model lines but no observations.
#
# The two measures need different estimators:
#   neonatal - one row per birth, neonatal_mortality is 0/1, so the cohort rate
#              is a plain mean.
#   under-5  - one row per child per AGE BIN, so a plain mean would give a
#              per-bin hazard (~1%) rather than the under-5 probability (~7.6%).
#              The comparable quantity is a discrete-time life table over the
#              eight bins: 5q0 = 1 - PROD(1 - deaths_b / at_risk_b).
#
# Cohorts observed through fewer than eight bins are KEPT (changed 2026-10-16).
# The model applies no completeness rule - it fits row-wise on child-age-bin
# observations - so excluding them here made the overlay stricter than the fit
# it is drawn against. Their life table runs over fewer bins and so understates
# 5q0; n_bins travels with each point to keep them identifiable.
XO_MIN_N = 100
XO_TRAIN_NN = f"{XO_ROOT}/neonatal_mortality/training_data/2026_10_08.01/data.parquet"
XO_TRAIN_CM = f"{XO_ROOT}/child_mortality/training_data/2026_10_07.01/data.parquet"
_xo_iso = (
    xo_hier.query("level == 3")
    .drop_duplicates("ihme_loc_id")
    .set_index("ihme_loc_id")["location_id"]
)

_nn_raw = pd.read_parquet(
    XO_TRAIN_NN, columns=["location_id", "birth_year", "neonatal_mortality"]
).dropna(subset=["location_id", "birth_year", "neonatal_mortality"])
xo_obs_nn = (
    _nn_raw.groupby(["location_id", "birth_year"], observed=True)
    .agg(rate=("neonatal_mortality", "mean"), n=("neonatal_mortality", "size"))
    .reset_index()
    .query("n >= @XO_MIN_N")
)

_cm_raw = pd.read_parquet(
    XO_TRAIN_CM, columns=["ihme_loc_id", "birth_year", "age_group", "child_mortality"]
).dropna(subset=["ihme_loc_id", "birth_year", "age_group", "child_mortality"])
_cm_bins = (
    _cm_raw.groupby(["ihme_loc_id", "birth_year", "age_group"], observed=True)
    .agg(deaths=("child_mortality", "sum"), at_risk=("child_mortality", "size"))
    .reset_index()
)
_cm_bins["surv"] = 1.0 - _cm_bins["deaths"] / _cm_bins["at_risk"]
_cm_cells = (
    _cm_bins.groupby(["ihme_loc_id", "birth_year"], observed=True)
    .agg(surv=("surv", "prod"), n_bins=("age_group", "nunique"), n=("at_risk", "max"))
    .reset_index()
)
_XO_N_BINS = _cm_raw["age_group"].nunique()
xo_obs_cm = _cm_cells[_cm_cells.n >= XO_MIN_N].copy()
xo_obs_cm["rate"] = 1.0 - xo_obs_cm["surv"]
xo_obs_cm["location_id"] = xo_obs_cm["ihme_loc_id"].map(_xo_iso)
xo_obs_cm = xo_obs_cm.dropna(subset=["location_id"])
xo_obs_cm["location_id"] = xo_obs_cm["location_id"].astype(int)
print(
    f"  observed: neonatal {len(xo_obs_nn):,} country-years "
    f"({xo_obs_nn.location_id.nunique()} countries); "
    f"under-5 {len(xo_obs_cm):,} country-years "
    f"({xo_obs_cm.location_id.nunique()} countries, life table over up to "
    f"{_XO_N_BINS} age bins; "
    f"{int((xo_obs_cm.n_bins < _XO_N_BINS).sum()):,} partial)"
)

# ---- payload --------------------------------------------------------------
_xo_years = sorted(xo_m.year_id.unique())
_xo_ids = sorted(xo_bad.location_id.unique())
_xo_locs = {}
for _lid in _xo_ids:
    _g = xo_m[xo_m.location_id == _lid].set_index("year_id").reindex(_xo_years)
    _ov = _g["over"].fillna(False).astype(bool).tolist()
    _yr_over = [int(y) for y, o in zip(_xo_years, _ov) if o]
    _fin = _g[_g["over"].fillna(False) & (_g.adjusted_model_value_cm > 0)]
    _ratio = (
        float((_fin.adjusted_model_value_nn / _fin.adjusted_model_value_cm).max())
        if len(_fin)
        else None
    )
    _r = lambda s: [None if pd.isna(v) else round(float(v), 7) for v in _g[s]]
    _xo_locs[str(_lid)] = {
        "n": str(_xo_nm.get(_lid)),
        "lv": int(_xo_lv.get(_lid)),
        "lvn": _XO_LVN.get(int(_xo_lv.get(_lid)), "Other"),
        "sr": (None if pd.isna(_xo_sr.get(_lid)) else str(_xo_sr.get(_lid))),
        "onset": int(min(_yr_over)),
        "nyears": len(_yr_over),
        # years where under-5 was clipped to zero while neonatal was positive
        "nzero": int(
            ((_g["over"].fillna(False)) & (_g.adjusted_model_value_cm <= 0)).sum()
        ),
        "ratio": None if _ratio is None else round(_ratio, 2),
        "over": _ov,
        "nn": _r("adjusted_model_value_nn"),
        "cm": _r("adjusted_model_value_cm"),
        "nn_pre": _r("model_value_nn"),
        "cm_pre": _r("model_value_cm"),
        "obs_nn": [
            [int(t.birth_year), round(float(t.rate), 7), int(t.n)]
            for t in xo_obs_nn[xo_obs_nn.location_id == _lid].itertuples()
        ],
        "obs_cm": [
            [int(t.birth_year), round(float(t.rate), 7), int(t.n)]
            for t in xo_obs_cm[xo_obs_cm.location_id == _lid].itertuples()
        ],
        "nn_gbd": _r("gbd_value_nn"),
        "cm_gbd": _r("gbd_value_cm"),
    }

_xo_by_lv = xo_bad.groupby(xo_bad.location_id.map(_xo_lv)).location_id.nunique()
_xo_zero = int((xo_bad.adjusted_model_value_cm <= 0).sum())
_xo_count_by_year = xo_bad.groupby("year_id").location_id.nunique()
XO_PAYLOAD = {
    "nn": {"measure": XO_NN[0].replace("_", " "), "rv": XO_NN[1]},
    "cm": {"measure": XO_CM[0].replace("_", " "), "rv": XO_CM[1]},
    "years": [int(y) for y in _xo_years],
    "minN": XO_MIN_N,
    "locs": _xo_locs,
    "summary": {
        "cells": int(len(xo_bad)),
        "totalCells": int(len(xo_m)),
        "locs": int(xo_bad.location_id.nunique()),
        "totalLocs": int(xo_m.location_id.nunique()),
        "byLevel": {_XO_LVN.get(int(k), str(k)): int(v) for k, v in _xo_by_lv.items()},
        "zeroCells": _xo_zero,
        "onsetMin": int(min(v["onset"] for v in _xo_locs.values())),
        "onsetMedian": int(np.median([v["onset"] for v in _xo_locs.values()])),
        "withObs": int(sum(1 for v in _xo_locs.values() if v["obs_nn"] or v["obs_cm"])),
        "curve": [
            [int(y), int(_xo_count_by_year.get(y, 0))]
            for y in (2000, 2010, 2023, 2030, 2040, 2050, 2075, 2100)
        ],
    },
}
print(
    f"  by level: {XO_PAYLOAD['summary']['byLevel']}; "
    f"{_xo_zero:,} of those cells have under-5 clipped to exactly 0; "
    f"onset median {XO_PAYLOAD['summary']['onsetMedian']}"
)

XO_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Neonatal above under-5</title>
<style>
:root{color-scheme:light;
  --plane:#f9f9f7; --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781;
  --grid:#e1e0d9; --axis:#c3c2b7; --ring:rgba(11,11,11,.10);
  --u5:#2a78d6; --nn:#eb6834; --accent:#2a78d6; --flag:rgba(137,135,129,.16);}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])){
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --u5:#3987e5; --nn:#d95926; --accent:#3987e5; --flag:rgba(137,135,129,.22);}}
:root[data-theme="dark"]{color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --u5:#3987e5; --nn:#d95926; --accent:#3987e5; --flag:rgba(137,135,129,.22);}
*{box-sizing:border-box}
body{margin:0;background:var(--plane);color:var(--ink);
  font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
header{padding:14px 16px;border-bottom:1px solid var(--ring);display:flex;
  align-items:baseline;gap:12px;flex-wrap:wrap}
h1{font-size:17px;margin:0;font-weight:650}
.sub{color:var(--ink2);font-size:12.5px}
.spacer{flex:1}
button,select,input{font:inherit;color:var(--ink);background:var(--surface);
  border:1px solid var(--ring);border-radius:7px;padding:6px 10px}
button{cursor:pointer} button:hover{border-color:var(--muted)}
button[aria-pressed="true"]{background:var(--accent);border-color:var(--accent);color:#fff}
.wrap{display:grid;grid-template-columns:300px minmax(0,1fr);gap:16px;padding:16px;
  align-items:start}
.panel{background:var(--surface);border:1px solid var(--ring);border-radius:11px}
#side{padding:12px;position:sticky;top:16px;max-height:calc(100vh - 32px);
  display:flex;flex-direction:column}
#q{width:100%;margin-bottom:8px}
#list{flex:1;min-height:0;overflow:auto;margin:0;padding:0;list-style:none}
#list li{padding:6px 9px;border-radius:7px;cursor:pointer;display:flex;gap:8px;
  align-items:baseline}
#list li:hover{background:var(--plane)}
#list li[aria-selected="true"]{background:var(--accent);color:#fff}
#list li[aria-selected="true"] .meta2{color:#fff;opacity:.85}
.meta2{color:var(--muted);font-size:11px;margin-left:auto;white-space:nowrap;
  font-variant-numeric:tabular-nums}
#list .grp{padding:10px 9px 3px;font-size:11px;letter-spacing:.06em;
  text-transform:uppercase;color:var(--muted);font-weight:650;cursor:default}
#list .grp:first-child{padding-top:2px}
#list .grp .cnt{text-transform:none;letter-spacing:0;font-weight:400}
#main{padding:14px 16px 18px}
h2{font-size:16px;margin:0 0 2px}
.meta{color:var(--ink2);font-size:12.5px;margin-bottom:10px}
.row{display:flex;gap:16px;align-items:center;flex-wrap:wrap;margin-bottom:8px}
.ctl{display:flex;gap:6px;align-items:center}
.ctl label{color:var(--ink2);font-size:12.5px}
#legend{display:flex;flex-wrap:wrap;gap:6px 14px;margin:10px 0 2px}
.lg{display:flex;gap:7px;align-items:center;border:0;background:none;padding:3px 5px;
  border-radius:6px;color:var(--ink2);font-size:12.5px;cursor:pointer}
.lg:hover{background:var(--plane)} .lg[aria-pressed="false"]{opacity:.38}
#chart{width:100%;height:auto;display:block;touch-action:none}
#tip{position:fixed;pointer-events:none;z-index:9;background:var(--surface);
  border:1px solid var(--ring);border-radius:9px;padding:9px 11px;font-size:12.5px;
  box-shadow:0 6px 22px rgba(0,0,0,.14);opacity:0;transition:opacity .08s}
#tip table{border-collapse:collapse;margin-top:5px}
#tip td{padding:1px 0} #tip td.v{text-align:right;padding-left:12px;
  font-variant-numeric:tabular-nums}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;
  padding:0 16px 16px}
.card{background:var(--surface);border:1px solid var(--ring);border-radius:11px;
  padding:12px 14px}
.card h3{margin:0 0 4px;font-size:12px;font-weight:650;color:var(--ink2)}
.big{font-size:23px;font-weight:650;font-variant-numeric:tabular-nums}
.card p{margin:3px 0 0;font-size:11.5px;color:var(--muted)}
table.t{width:100%;border-collapse:collapse;font-size:12.5px;
  font-variant-numeric:tabular-nums}
table.t th,table.t td{padding:3px 7px;text-align:right;border-bottom:1px solid var(--grid)}
table.t th:first-child,table.t td:first-child{text-align:left}
table.t thead th{color:var(--ink2);font-weight:600}
#tblwrap{max-height:60vh;overflow:auto;margin-top:10px}
.hidden{display:none !important}
footer{color:var(--muted);font-size:11.5px;padding:0 16px 22px;line-height:1.6}
@media (max-width:860px){.wrap{grid-template-columns:1fr}#side{position:static}
  #list{max-height:220px}}
</style></head><body>
<header><h1>Neonatal mortality above under-5 mortality</h1>
<span class="sub">neonatal __NNRV__ vs under-5 __CMRV__ &middot; shifted models
&middot; countries and subnationals only</span>
<span class="spacer"></span>
<button id="view" aria-pressed="false">Table</button><button id="theme">Theme</button>
</header>
<div class="cards" id="cards"></div>
<div class="wrap">
  <nav class="panel" id="side">
    <input id="q" type="search" placeholder="Search affected locations&hellip;  ( / )"
           autocomplete="off" aria-label="Search">
    <ul id="list" role="listbox" aria-label="Affected locations"></ul>
  </nav>
  <section class="panel" id="main">
    <h2 id="locname">&nbsp;</h2><div class="meta" id="locmeta">&nbsp;</div>
    <div class="row">
      <div class="ctl"><label for="scale">Y axis</label>
        <select id="scale"><option value="lin">Linear</option>
          <option value="log">Log</option></select></div>
      <div class="ctl"><label for="sort">Order list by</label>
        <select id="sort"><option value="onset">Earliest crossover</option>
          <option value="nyears">Most years affected</option>
          <option value="name">Name</option></select></div>
    </div>
    <figure id="figwrap" style="margin:0">
      <svg id="chart" role="img" aria-label="Neonatal and under-5 mortality"></svg>
      <div id="legend"></div>
    </figure>
    <div id="tblwrap" class="hidden"><table class="t" id="tbl"></table></div>
  </section>
</div>
<div id="tip" role="tooltip"></div>
<footer>
  <div>Neonatal deaths are a subset of under-5 deaths, so the orange line can
  never legitimately sit above the blue one. GBD never does: across its own
  138,750 location-year-sex cells the neonatal share of under-5 peaks at 0.81.
  The two measures are fitted as independent models with nothing linking them,
  so the containment is not enforced anywhere in the pipeline.</div>
  <div>Only countries and subnationals are shown. Global, super-region and
  region lines are built by rolling these up, so a crossover there follows from
  the places beneath it rather than being a separate occurrence.</div>
  <div>Points are observed survey rates from the training extracts, country
  level only and plotted by BIRTH year. Neonatal is a plain cohort mean; under-5
  is a discrete-time life table over the eight age bins, since that extract has
  one row per child per bin and a plain mean would give a per-bin hazard instead
  of the under-5 probability. Point size grows with sample size; cells with fewer
  than __MINN__ births are dropped.</div>
  <div>Shaded bands mark the years where it breaks. Where the under-5 line sits
  flat at zero it has been clipped there by the residual step, so the ratio is
  undefined rather than large.</div>
</footer>
<script>
const D = __PAYLOAD__;
const YEARS = D.years;
const SER = [
  {k:'cm', l:'Under-5, shifted model', c:'--u5', d:'none', w:2.5},
  {k:'nn', l:'Neonatal, shifted model', c:'--nn', d:'none', w:2.5},
  {k:'cm_pre', l:'Under-5, original model', c:'--u5', d:'7 4', w:1.8},
  {k:'nn_pre', l:'Neonatal, original model', c:'--nn', d:'7 4', w:1.8},
  // GBD is observed reference, not a model stage, so it moves off the series
  // hues onto the ink channel: solid black for under-5, solid dark grey for
  // neonatal. Both invert in dark mode (white / light grey) or they vanish.
  // Separation checked: dE(ink,ink2) 28.5 light / 18.9 dark, and >=21.8 against
  // both series hues in both modes.
  {k:'cm_gbd', l:'Under-5, GBD', c:'--ink', d:'none', w:1.8},
  {k:'nn_gbd', l:'Neonatal, GBD', c:'--ink2', d:'none', w:1.8},
];
// observed survey rates; country pages only, so these are absent on subnationals
const OBS = [
  {k:'obs_cm', l:'Under-5, survey (life table)', c:'--u5'},
  {k:'obs_nn', l:'Neonatal, survey', c:'--nn'},
];
const obsOn = k => !hidden.has(k);
const obsPts = k => (D.locs[cur][k]||[]);
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const esc = s => String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const fmt = v => v==null ? '—' : (100*v).toFixed(3)+'%';
let ids = Object.keys(D.locs), cur = ids[0], hidden = new Set(), filtered = ids, hi = 0, geom=null;

function sortIds(){
  const by = document.getElementById('sort').value;
  ids.sort((a,b)=>{const A=D.locs[a],B=D.locs[b];
    if(by==='onset') return A.onset-B.onset || B.nyears-A.nyears;
    if(by==='nyears') return B.nyears-A.nyears || A.onset-B.onset;
    return A.n.localeCompare(B.n);});
}
const hasObs = i => (D.locs[i].obs_nn.length>0 || D.locs[i].obs_cm.length>0);
function renderList(){
  const q=document.getElementById('q').value.trim().toLowerCase();
  const match = ids.filter(i=>{const L=D.locs[i];
    return !q || L.n.toLowerCase().includes(q) || (L.sr||'').toLowerCase().includes(q);});
  // Two sections. Survey data is coded to country level, so "without" is every
  // subnational plus the countries the extracts never covered - the pages where
  // the model lines cannot be checked against an observation.
  const withObs = match.filter(hasObs), noObs = match.filter(i=>!hasObs(i));
  filtered = withObs.concat(noObs);           // keyboard order follows the render
  const row = (i,k)=>{const L=D.locs[i];
    return `<li role="option" data-id="${i}" data-k="${k}" aria-selected="${i===cur}">
      <span>${esc(L.n)}</span><span class="meta2">${L.onset} &middot; ${L.nyears}y</span></li>`;};
  let out='', k=0;
  if(withObs.length){
    out += `<li class="grp">With survey data <span class="cnt">(${withObs.length})</span></li>`;
    withObs.forEach(i=>{out+=row(i,k++);});
  }
  if(noObs.length){
    out += `<li class="grp">No survey data <span class="cnt">(${noObs.length})</span></li>`;
    noObs.forEach(i=>{out+=row(i,k++);});
  }
  document.getElementById('list').innerHTML = out || '<li>No match</li>';
}
document.getElementById('list').addEventListener('click',e=>{
  const li=e.target.closest('li'); if(!li||!li.dataset.id) return;
  cur=li.dataset.id; hi=+li.dataset.k; location.hash='loc='+cur; draw();
});
const qEl=document.getElementById('q');
qEl.addEventListener('input',()=>{hi=0;renderList();});
qEl.addEventListener('keydown',e=>{
  if(e.key==='ArrowDown'||e.key==='ArrowUp'){e.preventDefault();
    hi=Math.min(filtered.length-1,Math.max(0,hi+(e.key==='ArrowDown'?1:-1)));
    const li=document.querySelector(`#list li[data-k="${hi}"]`);
    if(li){document.querySelectorAll('#list li').forEach(x=>x.setAttribute('aria-selected','false'));
      li.setAttribute('aria-selected','true');li.scrollIntoView({block:'nearest'});}}
  else if(e.key==='Enter'&&filtered[hi]){cur=filtered[hi];location.hash='loc='+cur;draw();}
});
document.addEventListener('keydown',e=>{
  if(e.key==='/'&&document.activeElement!==qEl){e.preventDefault();qEl.focus();qEl.select();}});

const svg=document.getElementById('chart'), tip=document.getElementById('tip');
const NS='http://www.w3.org/2000/svg';
const el=(t,a)=>{const n=document.createElementNS(NS,t);for(const k in a)n.setAttribute(k,a[k]);return n;};
const vis = () => SER.filter(s=>!hidden.has(s.k));

function draw(){
  const L=D.locs[cur];
  document.getElementById('locname').textContent=L.n;
  const ratio = L.ratio==null ? 'under-5 clipped to zero, ratio undefined'
    : `peak ratio ${L.ratio.toFixed(2)}x`;
  document.getElementById('locmeta').textContent =
    [L.lvn, L.sr||null, `first crossover ${L.onset}`, `${L.nyears} years affected`,
     ratio, 'location_id '+cur].filter(Boolean).join('  ·  ');
  document.querySelectorAll('#list li').forEach(li=>
    li.setAttribute('aria-selected',String(li.dataset.id===cur)));
  renderLegend(); drawChart(L); renderTable(L);
}
function drawChart(L){
  const W=svg.clientWidth||900, H=Math.max(330,Math.min(500,W*0.44));
  const m={t:14,r:18,b:34,l:62};
  svg.setAttribute('viewBox',`0 0 ${W} ${H}`); svg.setAttribute('height',H);
  svg.textContent='';
  const logy=document.getElementById('scale').value==='log';
  let vals=[]; vis().forEach(s=>(L[s.k]||[]).forEach(v=>{
    if(v!=null&&(!logy||v>0)) vals.push(v);}));
  OBS.forEach(o=>{ if(obsOn(o.k)) (L[o.k]||[]).forEach(([,r])=>{
    if(r!=null&&(!logy||r>0)) vals.push(r);});});
  if(!vals.length) vals=[0,1e-3];
  let lo,hiV=Math.max(...vals);
  if(logy){lo=Math.min(...vals); if(hiV<=lo)hiV=lo*1.5+1e-9; lo*=0.85; hiV*=1.15;}
  else {lo=Math.min(0,Math.min(...vals)); const p=(hiV-lo)*0.06; hiV+=p; if(lo<0)lo-=p;}
  const x=y=>m.l+(y-YEARS[0])/((YEARS[YEARS.length-1]-YEARS[0])||1)*(W-m.l-m.r);
  const Y=v=>{if(v==null)return null;
    const f=logy?(Math.log(Math.max(v,1e-12))-Math.log(lo))/(Math.log(hiV)-Math.log(lo))
                :(v-lo)/(hiV-lo);
    return m.t+(1-f)*(H-m.t-m.b);};
  geom={x,Y,m,W,H,L};
  // shaded bands first, so lines and gridlines sit above them
  let i=0;
  while(i<L.over.length){
    if(L.over[i]){let j=i; while(j+1<L.over.length&&L.over[j+1])j++;
      const x0=x(YEARS[i]), x1=x(YEARS[j]);
      svg.appendChild(el('rect',{x:x0,y:m.t,width:Math.max(1.5,x1-x0),
        height:H-m.t-m.b,fill:css('--flag')}));
      i=j+1;} else i++;
  }
  (logy?logTicks(lo,hiV):linTicks(lo,hiV,5)).forEach(t=>{
    const yy=Y(t); if(yy==null||yy<m.t-1||yy>H-m.b+1)return;
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:yy,y2:yy,stroke:css('--grid'),'stroke-width':1}));
    const lb=el('text',{x:m.l-8,y:yy+4,'text-anchor':'end','font-size':11,fill:css('--muted')});
    lb.textContent=(100*t).toFixed(t<0.01?2:(t<0.1?1:0))+'%'; svg.appendChild(lb);});
  for(let yv=YEARS[0]; yv<=YEARS[YEARS.length-1]; yv+=20){
    const lb=el('text',{x:x(yv),y:H-m.b+18,'text-anchor':'middle','font-size':11,fill:css('--muted')});
    lb.textContent=yv; svg.appendChild(lb);}
  svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:H-m.b,y2:H-m.b,stroke:css('--axis'),'stroke-width':1}));
  if(!logy&&lo<0){const z=Y(0);
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:z,y2:z,stroke:css('--axis'),'stroke-width':1.5}));}
  // observed points sit BELOW the model lines; radius grows with sample size
  OBS.forEach(o=>{ if(!obsOn(o.k)) return;
    (L[o.k]||[]).forEach(([yr,r,n])=>{
      if(r==null||(logy&&r<=0)) return;
      const yy=Y(r); if(yy==null||yr<YEARS[0]||yr>YEARS[YEARS.length-1]) return;
      svg.appendChild(el('circle',{cx:x(yr),cy:yy,r:Math.max(3,Math.min(7,Math.sqrt(n)/14)),
        fill:css(o.c),'fill-opacity':0.4,stroke:css(o.c),'stroke-width':1.2}));});});
  vis().forEach(s=>{const arr=L[s.k]||[]; let d='',pen=false;
    YEARS.forEach((yv,k)=>{const v=arr[k],yy=Y(v);
      if(v==null||yy==null||(logy&&v<=0)){pen=false;return;}
      d+=(pen?'L':'M')+x(yv).toFixed(1)+' '+yy.toFixed(1)+' ';pen=true;});
    if(d)svg.appendChild(el('path',{d,fill:'none',stroke:css(s.c),'stroke-width':s.w,
      'stroke-linecap':'round','stroke-linejoin':'round',
      ...(s.d!=='none'?{'stroke-dasharray':s.d}:{})}));});
  const cross=el('g',{id:'cross',visibility:'hidden'}); svg.appendChild(cross);
  cross.appendChild(el('line',{id:'cl',y1:m.t,y2:H-m.b,stroke:css('--muted'),'stroke-width':1}));
  const hit=el('rect',{x:m.l,y:m.t,width:Math.max(0,W-m.l-m.r),
    height:Math.max(0,H-m.t-m.b),fill:'transparent'});
  svg.appendChild(hit);
  hit.addEventListener('pointermove',onMove);
  hit.addEventListener('pointerleave',()=>{cross.setAttribute('visibility','hidden');tip.style.opacity=0;});
}
function linTicks(lo,hi,n){const raw=(hi-lo)/n,mag=Math.pow(10,Math.floor(Math.log10(raw||1e-9)));
  const st=[1,2,2.5,5,10].map(v=>v*mag).find(v=>v>=raw)||mag*10;const o=[];
  for(let v=Math.ceil(lo/st)*st;v<=hi;v+=st)o.push(v);return o;}
function logTicks(lo,hi){const o=[];
  for(let e=Math.floor(Math.log10(lo));e<=Math.ceil(Math.log10(hi));e++)
    for(const mm of [1,2,5]){const v=mm*Math.pow(10,e);if(v>=lo&&v<=hi)o.push(v);}
  return o.length?o:[lo,hi];}
function onMove(ev){
  if(!geom)return;
  const r=svg.getBoundingClientRect(), sc=geom.W/r.width, px=(ev.clientX-r.left)*sc;
  let best=YEARS[0],bd=1e9;
  YEARS.forEach(y=>{const d=Math.abs(geom.x(y)-px);if(d<bd){bd=d;best=y;}});
  const i=YEARS.indexOf(best), L=geom.L;
  const cross=svg.querySelector('#cross'); cross.setAttribute('visibility','visible');
  const cl=cross.querySelector('#cl'); cl.setAttribute('x1',geom.x(best)); cl.setAttribute('x2',geom.x(best));
  cross.querySelectorAll('circle').forEach(c=>c.remove());
  let rows='';
  vis().forEach(s=>{const v=(L[s.k]||[])[i]; if(v==null)return;
    const yy=geom.Y(v); if(yy==null)return;
    cross.appendChild(el('circle',{cx:geom.x(best),cy:yy,r:4,fill:css(s.c),
      stroke:css('--surface'),'stroke-width':2}));
    rows+=`<tr><td><svg width="16" height="8"><line x1="0" y1="4" x2="16" y2="4"
      stroke="${css(s.c)}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''}/></svg> ${esc(s.l)}</td>
      <td class="v">${fmt(v)}</td></tr>`;});
  OBS.forEach(o=>{ if(!obsOn(o.k)) return;
    const hit=(L[o.k]||[]).find(([yr])=>yr===best); if(!hit) return;
    rows+=`<tr><td><svg width="16" height="8"><circle cx="8" cy="4" r="3.5"
      fill="${css(o.c)}" fill-opacity="0.4" stroke="${css(o.c)}" stroke-width="1.2"/></svg>
      ${esc(o.l)} <span style="color:var(--muted)">n=${hit[2].toLocaleString()}</span></td>
      <td class="v">${fmt(hit[1])}</td></tr>`;});
  const flag = L.over[i] ? ' &middot; <b>neonatal above under-5</b>' : '';
  tip.innerHTML=`<b>${esc(L.n)} &middot; ${best}</b>${flag}<table>${rows}</table>`;
  tip.style.opacity=1;
  const tw=tip.offsetWidth,th=tip.offsetHeight;
  let lft=ev.clientX+16; if(lft+tw>innerWidth-8)lft=ev.clientX-tw-16;
  tip.style.left=Math.max(8,lft)+'px';
  tip.style.top=Math.min(Math.max(8,ev.clientY-th/2),innerHeight-th-8)+'px';
}
function renderLegend(){
  const b=document.getElementById('legend'); b.textContent='';
  SER.forEach(s=>{const x=document.createElement('button');
    x.className='lg'; x.setAttribute('aria-pressed',String(!hidden.has(s.k)));
    x.innerHTML=`<svg width="22" height="10"><line x1="0" y1="5" x2="22" y2="5"
      stroke="${css(s.c)}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''} stroke-linecap="round"/></svg>
      <span>${esc(s.l)}</span>`;
    x.onclick=()=>{hidden.has(s.k)?hidden.delete(s.k):hidden.add(s.k);draw();};
    b.appendChild(x);});
  OBS.forEach(o=>{const x=document.createElement('button');
    x.className='lg'; x.setAttribute('aria-pressed',String(obsOn(o.k)));
    x.innerHTML=`<svg width="22" height="10"><circle cx="11" cy="5" r="4"
      fill="${css(o.c)}" fill-opacity="0.4" stroke="${css(o.c)}" stroke-width="1.2"/></svg>
      <span>${esc(o.l)}</span>`;
    x.onclick=()=>{hidden.has(o.k)?hidden.delete(o.k):hidden.add(o.k);draw();};
    b.appendChild(x);});
  const t=document.createElement('span');
  t.className='lg'; t.style.cursor='default';
  t.innerHTML=`<svg width="22" height="10"><rect x="0" y="0" width="22" height="10"
    fill="${css('--flag')}"/></svg><span>neonatal above under-5</span>`;
  b.appendChild(t);
}
function renderTable(L){
  let h='<thead><tr><th>Year</th>'+SER.map(s=>`<th>${esc(s.l)}</th>`).join('')
       +'<th>Flag</th></tr></thead><tbody>';
  YEARS.forEach((y,i)=>{h+=`<tr><td>${y}</td>`
    +SER.map(s=>`<td>${fmt((L[s.k]||[])[i])}</td>`).join('')
    +`<td>${L.over[i]?'above':''}</td></tr>`;});
  document.getElementById('tbl').innerHTML=h+'</tbody>';
}
function renderCards(){
  const U=D.summary;
  const lv=Object.entries(U.byLevel).map(([k,v])=>`${v} ${k.toLowerCase()}`).join(', ');
  const curve=U.curve.map(([y,n])=>`${y}: ${n}`).join(' &middot; ');
  document.getElementById('cards').innerHTML = `
   <div class="card"><h3>Location-years affected</h3>
     <div class="big">${U.cells.toLocaleString()}</div>
     <p>of ${U.totalCells.toLocaleString()} compared</p></div>
   <div class="card"><h3>Locations affected</h3>
     <div class="big">${U.locs}</div><p>of ${U.totalLocs} modelled &middot; ${lv}</p></div>
   <div class="card"><h3>Under-5 clipped to zero</h3>
     <div class="big">${Math.round(100*U.zeroCells/U.cells)}%</div>
     <p>${U.zeroCells.toLocaleString()} cells where under-5 is exactly 0</p></div>
   <div class="card"><h3>First crossover</h3>
     <div class="big">${U.onsetMin}</div><p>median onset ${U.onsetMedian}</p></div>
   <div class="card"><h3>Checkable against survey data</h3>
     <div class="big">${U.withObs}</div>
     <p>${U.locs-U.withObs} have no observations at their level</p></div>
   <div class="card" style="grid-column:1/-1"><h3>Locations above, by year</h3>
     <p style="font-size:12.5px;color:var(--ink2)">${curve}</p></div>`;
}
document.getElementById('view').onclick=e=>{
  const showTable=e.target.getAttribute('aria-pressed')!=='true';
  e.target.setAttribute('aria-pressed',String(showTable));
  document.getElementById('figwrap').classList.toggle('hidden',showTable);
  document.getElementById('tblwrap').classList.toggle('hidden',!showTable);};
document.getElementById('theme').onclick=()=>{
  const now=document.documentElement.getAttribute('data-theme');
  const dark=now?now==='dark':matchMedia('(prefers-color-scheme: dark)').matches;
  document.documentElement.setAttribute('data-theme',dark?'light':'dark');
  draw(); renderCards();};
document.getElementById('scale').onchange=draw;
document.getElementById('sort').onchange=()=>{sortIds();renderList();};
addEventListener('resize',()=>{if(geom)drawChart(D.locs[cur]);});
const hashId=(location.hash.match(/loc=(\d+)/)||[])[1];
sortIds(); if(hashId&&D.locs[hashId])cur=hashId; else cur=ids[0];
renderList(); renderCards(); draw();
</script></body></html>
"""

_xo_doc = (
    XO_HTML.replace("__NNRV__", html.escape(XO_NN[1]))
    .replace("__CMRV__", html.escape(XO_CM[1]))
    .replace("__MINN__", str(XO_MIN_N))
    .replace(
        "__PAYLOAD__",
        json.dumps(XO_PAYLOAD, separators=(",", ":")).replace("</", "<\\/"),
    )
)
Path(XO_PATH).write_text(_xo_doc, encoding="utf-8")
print(f"Saved {XO_PATH} ({len(_xo_doc) / 1e6:.2f} MB)")

6,255 location-years where neonatal > under-5, across 174 of 484 countries and subnationals (aggregates excluded)
  observed: neonatal 879 country-years (60 countries); under-5 920 country-years (60 countries, life table over 8 age bins)
  by level: {'Country': 99, 'Subnational': 75}; 3,104 of those cells have under-5 clipped to exactly 0; onset median 2054
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/neonatal_vs_under5_crossover.html (1.08 MB)


In [ ]:
# Neonatal above under-5: shifted and original models side by side ###########
# Countries only, crossings counted after XP_FROM. Each country lands in one of
# three sections:
#   both      - the shifted AND the original model both put neonatal above
#   original  - only the pre-shift inference output does
#   shifted   - only the post-residual output does
# which separates a problem the residual step created from one it inherited.
import html as _xp_html
import json as _xp_json

XP_ROOT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition"
XP_NN = ("neonatal_mortality", "2026_10_08.01")
XP_CM = ("child_mortality", "2026_10_01.01")
XP_FROM = 2023
XP_LEVEL = 3
XP_PATH = f"{PLOT_OUTPUT}neonatal_vs_under5_crossover_original.html"
XP_POP_FUT = "/mnt/share/forecasting/data/32/future/population/future_population_s130v41/population.nc"
XP_POP_PAST = "/mnt/share/forecasting/data/16/past/population/20250603_etl_run_id_417/population.nc"
XP_AGE_AGGREGATES = {42: [2, 3], 1: [2, 3, 388, 389, 238, 34]}
XP_VALS = ["adjusted_model_value", "model_value", "gbd_value"]
XP_DRAW_SCENARIO = "ssp245"
XP_CI = (2.5, 97.5)
XP_MIN_N = 100
XP_TRAIN_NN = f"{XP_ROOT}/neonatal_mortality/training_data/2026_10_08.01/data.parquet"
XP_TRAIN_CM = f"{XP_ROOT}/child_mortality/training_data/2026_10_07.01/data.parquet"

_xp_hier = pd.read_parquet(f"{XP_ROOT}/input/fhs_hierarchy.parquet")
_xp_det = _xp_hier[_xp_hier["most_detailed"] == 1][
    ["location_id", "path_to_top_parent"]
]
_xp_nm = _xp_hier.set_index("location_id")["location_name"]
_xp_lv = _xp_hier.set_index("location_id")["level"]
_xp_sr = _xp_hier.set_index("location_id")["super_region_name"]


def _xp_pop(ages, sexes, years):
    detailed = sorted({a for g in ages for a in XP_AGE_AGGREGATES.get(g, [g])})
    out = []
    for _p, _sel in ((XP_POP_PAST, {}), (XP_POP_FUT, {"scenario": 130})):
        _d = (
            xr.open_dataset(_p)
            .sel(age_group_id=detailed, sex_id=sexes, **_sel)
            .to_dataframe()
            .reset_index()
        )
        _v = "population" if "population" in _d.columns else "draws"
        _k = ["location_id", "year_id", "age_group_id", "sex_id"]
        _d = (
            _d.groupby(_k, as_index=False)[_v].mean().rename(columns={_v: "population"})
        )
        _parts = []
        for _g in ages:
            _s = _d[_d.age_group_id.isin(XP_AGE_AGGREGATES.get(_g, [_g]))].copy()
            _s["age_group_id"] = _g
            _parts.append(_s.groupby(_k, as_index=False)["population"].sum())
        out.append(pd.concat(_parts))
    _P = pd.concat(out).drop_duplicates(
        ["location_id", "year_id", "age_group_id", "sex_id"], keep="last"
    )
    return _P[_P.year_id.isin(years)]


def _xp_agg(measure, rv):
    _s = pd.read_parquet(f"{XP_ROOT}/{measure}/results/{rv}/sev_means.parquet")
    for _c in [c for c in _s.columns if c.startswith("adjusted_")]:
        _s[_c] = _s[_c].clip(0, 1)
    _ages = sorted(_s.index.get_level_values("age_group_id").unique())
    _sexes = sorted(_s.index.get_level_values("sex_id").unique())
    _years = sorted(_s.index.get_level_values("year_id").unique())
    _vals = [c for c in XP_VALS if c in _s.columns]
    _f = (
        _s.reset_index()
        .merge(_xp_det, on="location_id", how="inner")
        .merge(
            _xp_pop(_ages, _sexes, _years),
            on=["location_id", "year_id", "age_group_id", "sex_id"],
            how="inner",
        )
    )
    _f["agg_id"] = _f["path_to_top_parent"].str.split(",")
    _e = _f.explode("agg_id")
    _e["agg_id"] = _e["agg_id"].astype(int)
    for _c in _vals:
        _e[_c] = _e[_c] * _e["population"]
    _a = _e.groupby(["agg_id", "year_id"], as_index=False)[_vals + ["population"]].sum(
        min_count=1
    )
    for _c in _vals:
        _a[_c] = _a[_c] / _a["population"]
    return _a.rename(columns={"agg_id": "location_id"})[
        ["location_id", "year_id"] + _vals
    ]


def _xp_country_map():
    _p2t = _xp_hier.set_index("location_id")["path_to_top_parent"]
    _d = _xp_hier[_xp_hier.most_detailed == 1].location_id.tolist()

    def _c(i):
        for _a in (int(x) for x in str(_p2t.get(i, "")).split(",") if x):
            if _xp_lv.get(_a) == XP_LEVEL:
                return _a
        return None

    _m = pd.DataFrame({"location_id": _d, "cid": [_c(i) for i in _d]}).dropna()
    _m["cid"] = _m["cid"].astype(int)
    return _m


def _xp_draw_ci(measure, rv):
    """95% interval of the ORIGINAL model at country level.

    Each draw is population-weighted to the country first, then percentiles are
    taken across draws; percentiles taken before aggregating would understate
    the interval. Checked: the draw mean reproduces model_value to 8e-08.
    Years before 2024 run a single draw, so the interval is zero-width there.
    """
    _d = pd.read_parquet(
        f"{XP_ROOT}/{measure}/results/{rv}/{XP_DRAW_SCENARIO}.parquet"
    ).reset_index()
    _dc = [c for c in _d.columns if c.startswith("draw_")]
    _j = _d.merge(_xp_country_map(), on="location_id", how="inner").merge(
        _xp_pop(
            sorted(_d.age_group_id.unique()),
            sorted(_d.sex_id.unique()),
            sorted(_d.year_id.unique()),
        ),
        on=["location_id", "year_id", "age_group_id", "sex_id"],
        how="inner",
    )
    _w = _j[_dc].to_numpy() * _j["population"].to_numpy()[:, None]
    _a = pd.DataFrame(_w, columns=_dc)
    _a["cid"] = _j.cid.values
    _a["year_id"] = _j.year_id.values
    _a["population"] = _j.population.values
    _g = _a.groupby(["cid", "year_id"]).sum()
    _v = _g[_dc].to_numpy() / _g["population"].to_numpy()[:, None]
    return (
        pd.DataFrame(
            {
                "lo": np.percentile(_v, XP_CI[0], axis=1),
                "hi": np.percentile(_v, XP_CI[1], axis=1),
            },
            index=_g.index,
        )
        .reset_index()
        .rename(columns={"cid": "location_id"})
    )


xp_m = _xp_agg(*XP_NN).merge(
    _xp_agg(*XP_CM), on=["location_id", "year_id"], suffixes=("_nn", "_cm")
)
xp_m = xp_m[xp_m.location_id.map(_xp_lv) == XP_LEVEL]
xp_m = xp_m.merge(
    _xp_draw_ci(*XP_NN).rename(columns={"lo": "lo_nn", "hi": "hi_nn"}),
    on=["location_id", "year_id"],
    how="left",
).merge(
    _xp_draw_ci(*XP_CM).rename(columns={"lo": "lo_cm", "hi": "hi_cm"}),
    on=["location_id", "year_id"],
    how="left",
)
xp_m["over_s"] = (xp_m.adjusted_model_value_nn > xp_m.adjusted_model_value_cm) & (
    xp_m.year_id > XP_FROM
)
xp_m["over_o"] = (xp_m.model_value_nn > xp_m.model_value_cm) & (xp_m.year_id > XP_FROM)

# observed survey rates; reuse the companion cell's work when it has run
try:
    _xp_obs_nn, _xp_obs_cm = xo_obs_nn, xo_obs_cm
    print("  observed rates: reusing xo_obs_* from the cell above")
except NameError:
    _xp_iso = (
        _xp_hier.query("level == 3")
        .drop_duplicates("ihme_loc_id")
        .set_index("ihme_loc_id")["location_id"]
    )
    _n = pd.read_parquet(
        XP_TRAIN_NN, columns=["location_id", "birth_year", "neonatal_mortality"]
    ).dropna()
    _xp_obs_nn = (
        _n.groupby(["location_id", "birth_year"], observed=True)
        .agg(rate=("neonatal_mortality", "mean"), n=("neonatal_mortality", "size"))
        .reset_index()
        .query("n >= @XP_MIN_N")
    )
    _c = pd.read_parquet(
        XP_TRAIN_CM,
        columns=["ihme_loc_id", "birth_year", "age_group", "child_mortality"],
    ).dropna()
    _b = (
        _c.groupby(["ihme_loc_id", "birth_year", "age_group"], observed=True)
        .agg(deaths=("child_mortality", "sum"), at_risk=("child_mortality", "size"))
        .reset_index()
    )
    _b["surv"] = 1.0 - _b["deaths"] / _b["at_risk"]
    _cc = (
        _b.groupby(["ihme_loc_id", "birth_year"], observed=True)
        .agg(
            surv=("surv", "prod"), n_bins=("age_group", "nunique"), n=("at_risk", "max")
        )
        .reset_index()
    )
    _nb = _c["age_group"].nunique()
    # Partial cohorts are KEPT (changed 2026-10-16). run_training.py builds its
    # frame as full_training_data.loc[:, raw_variables] and drops only rows with
    # nulls - there is no cohort-completeness rule - so every right-censored
    # cohort reaches the fit. Verified for India against the fitted frame
    # child_mortality/models/2026_09_24.03/raw_with_predictions.parquet:
    # 6,611,145 training rows, 6,611,145 non-null, 6,611,145 in the fit, of
    # which 440,983 come from the 2018-2021 cohorts that hold only 7/6/5/3 bins.
    # Their life table runs over fewer bins and so understates 5q0; n_bins
    # travels with each point so the chart can mark them hollow.
    _xp_obs_cm = _cc[_cc.n >= XP_MIN_N].copy()
    _xp_obs_cm["rate"] = 1.0 - _xp_obs_cm["surv"]
    _xp_obs_cm["location_id"] = _xp_obs_cm["ihme_loc_id"].map(_xp_iso)
    _xp_obs_cm = _xp_obs_cm.dropna(subset=["location_id"])
    _xp_obs_cm["location_id"] = _xp_obs_cm["location_id"].astype(int)
    print("  observed rates: computed here (life table for under-5)")

# full-bin count, whichever branch above produced the frame
_XP_N_BINS = int(_xp_obs_cm["n_bins"].max())
_xp_partial = int((_xp_obs_cm.n_bins < _XP_N_BINS).sum())
print(
    f"  under-5 overlay: {len(_xp_obs_cm):,} country-years, "
    f"{_xp_partial:,} from partial cohorts (< {_XP_N_BINS} bins)"
)

# ---- sections and sort keys ------------------------------------------------
_xp_years = sorted(xp_m.year_id.unique())
_xp_flag = xp_m.groupby("location_id")[["over_s", "over_o"]].any()
_xp_ids = sorted(_xp_flag[_xp_flag.over_s | _xp_flag.over_o].index)
_XP_SECT = {
    "both": "Both models cross",
    "orig": "Original model only",
    "shift": "Shifted model only",
}
_xp_locs = {}
for _lid in _xp_ids:
    _g = xp_m[xp_m.location_id == _lid].set_index("year_id").reindex(_xp_years)
    _r = lambda s: [None if pd.isna(v) else round(float(v), 7) for v in _g[s]]
    _os = _g["over_s"].fillna(False).astype(bool)
    _oo = _g["over_o"].fillna(False).astype(bool)
    # size of the breach: how far neonatal sits ABOVE under-5, in the years it does
    _ds = (_g.adjusted_model_value_nn - _g.adjusted_model_value_cm)[_os].dropna()
    _do = (_g.model_value_nn - _g.model_value_cm)[_oo].dropna()
    _sect = "both" if (_os.any() and _oo.any()) else ("orig" if _oo.any() else "shift")
    _xp_locs[str(_lid)] = {
        "n": str(_xp_nm.get(_lid)),
        "lv": int(_xp_lv.get(_lid)),
        "sr": (None if pd.isna(_xp_sr.get(_lid)) else str(_xp_sr.get(_lid))),
        "sect": _sect,
        "smax": (None if _ds.empty else round(100 * float(_ds.max()), 4)),
        "smaxYear": (None if _ds.empty else int(_ds.idxmax())),
        "omax": (None if _do.empty else round(100 * float(_do.max()), 4)),
        "omaxYear": (None if _do.empty else int(_do.idxmax())),
        "over_s": _os.tolist(),
        "over_o": _oo.tolist(),
        "cm": _r("adjusted_model_value_cm"),
        "nn": _r("adjusted_model_value_nn"),
        "cm_pre": _r("model_value_cm"),
        "nn_pre": _r("model_value_nn"),
        "cm_gbd": _r("gbd_value_cm"),
        "nn_gbd": _r("gbd_value_nn"),
        "lo_cm": _r("lo_cm"),
        "hi_cm": _r("hi_cm"),
        "lo_nn": _r("lo_nn"),
        "hi_nn": _r("hi_nn"),
        "obs_nn": [
            [int(t.birth_year), round(float(t.rate), 7), int(t.n)]
            for t in _xp_obs_nn[_xp_obs_nn.location_id == _lid].itertuples()
        ],
        "obs_cm": [
            [int(t.birth_year), round(float(t.rate), 7), int(t.n), int(t.n_bins)]
            for t in _xp_obs_cm[_xp_obs_cm.location_id == _lid].itertuples()
        ],
    }
_xp_cnt = {k: sum(1 for v in _xp_locs.values() if v["sect"] == k) for k in _XP_SECT}
XP_PAYLOAD = {
    "nnrv": XP_NN[1],
    "cmrv": XP_CM[1],
    "from": XP_FROM,
    "scen": XP_DRAW_SCENARIO,
    "minN": XP_MIN_N,
    "nBins": _XP_N_BINS,
    "years": [int(y) for y in _xp_years],
    "sections": _XP_SECT,
    "locs": _xp_locs,
    "summary": {
        "locs": len(_xp_locs),
        "totalLocs": int(_xp_flag.shape[0]),
        "counts": _xp_cnt,
        "withObs": int(sum(1 for v in _xp_locs.values() if v["obs_nn"] or v["obs_cm"])),
    },
}
print(
    f"countries crossing after {XP_FROM}: {len(_xp_locs)} of {_xp_flag.shape[0]}  "
    f"(both {_xp_cnt['both']}, original only {_xp_cnt['orig']}, "
    f"shifted only {_xp_cnt['shift']}); {XP_PAYLOAD['summary']['withObs']} have survey data"
)
XP_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Original models crossing</title>
<style>
:root{color-scheme:light;
  --plane:#f9f9f7; --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781;
  --grid:#e1e0d9; --axis:#c3c2b7; --ring:rgba(11,11,11,.10);
  --u5:#2a78d6; --nn:#eb6834; --accent:#2a78d6; --flag:rgba(137,135,129,.16);}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])){
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --u5:#3987e5; --nn:#d95926; --accent:#3987e5; --flag:rgba(137,135,129,.22);}}
:root[data-theme="dark"]{color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --u5:#3987e5; --nn:#d95926; --accent:#3987e5; --flag:rgba(137,135,129,.22);}
*{box-sizing:border-box}
body{margin:0;background:var(--plane);color:var(--ink);
  font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
header{padding:14px 16px;border-bottom:1px solid var(--ring);display:flex;
  align-items:baseline;gap:12px;flex-wrap:wrap}
h1{font-size:17px;margin:0;font-weight:650}
.sub{color:var(--ink2);font-size:12.5px}.spacer{flex:1}
button,select,input{font:inherit;color:var(--ink);background:var(--surface);
  border:1px solid var(--ring);border-radius:7px;padding:6px 10px}
button{cursor:pointer}button:hover{border-color:var(--muted)}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;
  padding:14px 16px 0}
.card{background:var(--surface);border:1px solid var(--ring);border-radius:11px;padding:12px 14px}
.card h3{margin:0 0 4px;font-size:12px;font-weight:650;color:var(--ink2)}
.big{font-size:23px;font-weight:650;font-variant-numeric:tabular-nums}
.card p{margin:3px 0 0;font-size:11.5px;color:var(--muted)}
.wrap{display:grid;grid-template-columns:290px minmax(0,1fr);gap:16px;padding:16px;
  align-items:start}
.panel{background:var(--surface);border:1px solid var(--ring);border-radius:11px}
#side{padding:12px;position:sticky;top:16px;max-height:calc(100vh - 32px);
  display:flex;flex-direction:column}
#q{width:100%;margin-bottom:8px}
#list{flex:1;min-height:0;overflow:auto;margin:0;padding:0;list-style:none}
#list li{padding:6px 9px;border-radius:7px;cursor:pointer;display:flex;gap:8px;align-items:baseline}
#list li:hover{background:var(--plane)}
#list li[aria-selected="true"]{background:var(--accent);color:#fff}
#list li[aria-selected="true"] .m2{color:#fff;opacity:.85}
.m2{color:var(--muted);font-size:11px;margin-left:auto;white-space:nowrap;
  font-variant-numeric:tabular-nums}
#list .grp{padding:10px 9px 3px;font-size:11px;letter-spacing:.06em;
  text-transform:uppercase;color:var(--muted);font-weight:650;cursor:default}
#list .grp:first-child{padding-top:2px}
#list .grp .cnt{text-transform:none;letter-spacing:0;font-weight:400}
#main{padding:14px 16px 18px}
h2{font-size:16px;margin:0 0 2px}
.meta{color:var(--ink2);font-size:12.5px;margin-bottom:10px}
#legend{display:flex;flex-wrap:wrap;gap:6px 14px;margin:10px 0 2px}
.lg{display:flex;gap:7px;align-items:center;border:0;background:none;padding:3px 5px;
  border-radius:6px;color:var(--ink2);font-size:12.5px;cursor:pointer}
.lg:hover{background:var(--plane)}.lg[aria-pressed="false"]{opacity:.38}
#chart{width:100%;height:auto;display:block;touch-action:none}
#tip{position:fixed;pointer-events:none;z-index:9;background:var(--surface);
  border:1px solid var(--ring);border-radius:9px;padding:9px 11px;font-size:12.5px;
  box-shadow:0 6px 22px rgba(0,0,0,.14);opacity:0;transition:opacity .08s}
#tip table{border-collapse:collapse;margin-top:5px}#tip td{padding:1px 0}
#tip td.v{text-align:right;padding-left:12px;font-variant-numeric:tabular-nums}
footer{color:var(--muted);font-size:11.5px;padding:0 16px 22px;line-height:1.6}
@media (max-width:860px){.wrap{grid-template-columns:1fr}#side{position:static}
  #list{max-height:220px}}
</style></head><body>
<header><h1>Neonatal above under-5: shifted vs original</h1>
<span class="sub">neonatal __NNRV__ vs under-5 __CMRV__ &middot; national level
&middot; crossings after __FROM__</span>
<span class="spacer"></span><button id="theme">Theme</button></header>
<div class="cards" id="cards"></div>
<div class="wrap">
  <nav class="panel" id="side">
    <select id="sect" aria-label="Which crossing" style="width:100%;margin-bottom:6px"></select>
    <input id="q" type="search" placeholder="Search&hellip;  ( / )" aria-label="Search">
    <div style="font-size:11px;color:var(--muted);padding:0 2px 6px">
      ordered by the shifted model's worst breach</div>
    <ul id="list" role="listbox"></ul></nav>
  <section class="panel" id="main">
    <h2 id="locname">&nbsp;</h2><div class="meta" id="locmeta">&nbsp;</div>
    <svg id="chart" role="img" aria-label="Original models"></svg>
    <div id="legend"></div>
  </section>
</div>
<div id="tip" role="tooltip"></div>
<footer>
  <div>Neonatal deaths are a subset of under-5 deaths, so an orange line can
  never legitimately sit above its blue counterpart; GBD, in black and grey,
  never does. Solid lines are the shifted models, dashed are the original
  pre-shift inference output, points are observed survey rates. Hollow
  under-5 points are cohorts not yet observed through all
  <span id="nbins"></span> age bins: their life table runs over fewer bins and
  so sits below the true 5q0.</div>
  <div>Sections separate what the residual step <i>created</i> from what it
  <i>inherited</i>. Within a section, countries are ordered by how far the
  shifted model puts neonatal above under-5 at its worst year; the
  original-only section, having no shifted breach, falls back to the original
  model's worst year.</div>
  <div>Ribbons are the 2.5th&ndash;97.5th percentile of the 100 __SCEN__ draws,
  each draw population-weighted to the country before the percentiles are taken.
  They are narrow: from 2024 the median 95% width is about 1.4% of the central
  estimate, so most crossings are not explained by draw spread. Before 2024 the
  ribbon has zero width &mdash; historical years are run at a single draw, so
  there is no spread to show.</div>
  <div>Full-height shading marks years the shifted model inverts; the strip
  along the top marks the original model. Survey points are country level and
  plotted by birth year: neonatal is a cohort mean, under-5 a life table over
  eight age bins. National level only.</div>
</footer>
<script>
const D = __PAYLOAD__, YEARS = D.years;
const BAND=[{lo:'lo_cm',hi:'hi_cm',c:'--u5',l:'Under-5, 95% of draws'},
            {lo:'lo_nn',hi:'hi_nn',c:'--nn',l:'Neonatal, 95% of draws'}];
const bandOn=()=>!hidden.has('__band__');
const SER=[{k:'cm',l:'Under-5, shifted model',c:'--u5',d:'none',w:2.5},
           {k:'nn',l:'Neonatal, shifted model',c:'--nn',d:'none',w:2.5},
           {k:'cm_pre',l:'Under-5, original model',c:'--u5',d:'7 4',w:1.8},
           {k:'nn_pre',l:'Neonatal, original model',c:'--nn',d:'7 4',w:1.8},
           {k:'cm_gbd',l:'Under-5, GBD',c:'--ink',d:'none',w:1.8},
           {k:'nn_gbd',l:'Neonatal, GBD',c:'--muted',d:'none',w:1.8}];
const OBS=[{k:'obs_cm',l:'Under-5, survey (life table)',c:'--u5'},
           {k:'obs_nn',l:'Neonatal, survey',c:'--nn'}];
const obsOn=k=>!hidden.has(k);
const css=n=>getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const fmt=v=>v==null?'—':(100*v).toFixed(3)+'%';
let ids=Object.keys(D.locs), cur=ids[0], hidden=new Set(), filtered=ids, hi=0, geom=null;
let sect='all';
const SECT=['both','orig','shift'];
// Within a section, ordered by how far the SHIFTED model puts neonatal above
// under-5 at its worst year. The "original only" section has no shifted breach
// by definition, so it falls back to the original model's worst year.
const key=i=>{const L=D.locs[i];
  return L.smax!=null?L.smax:(L.omax!=null?L.omax:-Infinity);};
ids.sort((a,b)=>key(b)-key(a)||D.locs[a].n.localeCompare(D.locs[b].n));
function renderSect(){
  const el2=document.getElementById('sect');
  const n=k=>ids.filter(i=>D.locs[i].sect===k).length;
  el2.innerHTML = `<option value="all">All countries (${ids.length})</option>`
    + SECT.map(k=>`<option value="${k}">${esc(D.sections[k])} (${n(k)})</option>`).join('');
  el2.value = sect;
}
function renderList(){
  const q=document.getElementById('q').value.trim().toLowerCase();
  // one section at a time, chosen from the dropdown: scrolling the list no
  // longer silently crosses from one category into another
  filtered=ids.filter(i=>{const L=D.locs[i];
    return (sect==='all'||L.sect===sect)
      && (!q||L.n.toLowerCase().includes(q)||(L.sr||'').toLowerCase().includes(q));});
  document.getElementById('list').innerHTML=filtered.map((i,k)=>{const L=D.locs[i];
    const v=L.smax!=null?L.smax:L.omax;
    const tag=sect==='all'?` &middot; ${esc(D.sections[L.sect].replace(' models cross',' both').replace(' model only',''))}`:'';
    return `<li role="option" data-id="${i}" data-k="${k}" aria-selected="${i===cur}">
      <span>${esc(L.n)}</span><span class="m2">${
        v==null?'\u2014':'+'+v.toFixed(2)+' pp'}${tag}</span></li>`;
  }).join('')||'<li>No match</li>';
}
document.getElementById('list').addEventListener('click',e=>{
  const li=e.target.closest('li'); if(!li||!li.dataset.id)return;
  cur=li.dataset.id; hi=+li.dataset.k; location.hash='loc='+cur; draw();});
const qEl=document.getElementById('q');
qEl.addEventListener('input',()=>{hi=0;renderList();});
qEl.addEventListener('keydown',e=>{
  if(e.key==='ArrowDown'||e.key==='ArrowUp'){e.preventDefault();
    hi=Math.min(filtered.length-1,Math.max(0,hi+(e.key==='ArrowDown'?1:-1)));
    const li=document.querySelector(`#list li[data-k="${hi}"]`);
    if(li){document.querySelectorAll('#list li').forEach(x=>x.setAttribute('aria-selected','false'));
      li.setAttribute('aria-selected','true');li.scrollIntoView({block:'nearest'});}}
  else if(e.key==='Enter'&&filtered[hi]){cur=filtered[hi];location.hash='loc='+cur;draw();}});
document.addEventListener('keydown',e=>{
  if(e.key==='/'&&document.activeElement!==qEl){e.preventDefault();qEl.focus();qEl.select();}});
const svg=document.getElementById('chart'),tip=document.getElementById('tip');
const NS='http://www.w3.org/2000/svg';
const el=(t,a)=>{const n=document.createElementNS(NS,t);for(const k in a)n.setAttribute(k,a[k]);return n;};
const vis=()=>SER.filter(s=>!hidden.has(s.k));
function draw(){
  const L=D.locs[cur];
  document.getElementById('locname').textContent=L.n;
  document.getElementById('locmeta').textContent=[L.lvn,L.sr||null,
    `first crossing ${L.onset}`,`${L.nyears} years`,'location_id '+cur]
    .filter(Boolean).join('  ·  ');
  document.querySelectorAll('#list li').forEach(li=>
    li.setAttribute('aria-selected',String(li.dataset.id===cur)));
  renderLegend(); drawChart(L);
}
function drawChart(L){
  const W=svg.clientWidth||900,H=Math.max(320,Math.min(480,W*0.44));
  const m={t:14,r:18,b:34,l:62};
  svg.setAttribute('viewBox',`0 0 ${W} ${H}`);svg.setAttribute('height',H);svg.textContent='';
  let vals=[]; vis().forEach(s=>(L[s.k]||[]).forEach(v=>{if(v!=null)vals.push(v);}));
  if(bandOn())BAND.forEach(b=>[b.lo,b.hi].forEach(k=>(L[k]||[]).forEach(v=>{
    if(v!=null)vals.push(v);})));
  OBS.forEach(o=>{ if(obsOn(o.k))(L[o.k]||[]).forEach(([,r])=>{if(r!=null)vals.push(r);});});
  if(!vals.length)vals=[0,1e-3];
  let lo=Math.min(0,Math.min(...vals)),hiV=Math.max(...vals);
  const pad=(hiV-lo)*0.06; hiV+=pad; if(lo<0)lo-=pad;
  const x=y=>m.l+(y-YEARS[0])/((YEARS[YEARS.length-1]-YEARS[0])||1)*(W-m.l-m.r);
  const Y=v=>v==null?null:m.t+(1-(v-lo)/(hiV-lo))*(H-m.t-m.b);
  geom={x,Y,m,W,H,L};
  const bands=(arr,yy,hh)=>{let i=0;
    while(i<arr.length){ if(arr[i]){let j=i;while(j+1<arr.length&&arr[j+1])j++;
      svg.appendChild(el('rect',{x:x(YEARS[i]),y:yy,
        width:Math.max(1.5,x(YEARS[j])-x(YEARS[i])),height:hh,fill:css('--flag')}));
      i=j+1;} else i++; }};
  bands(L.over_s,m.t,H-m.t-m.b);     // full height: shifted model crossing
  bands(L.over_o,m.t,7);             // top strip: original model crossing
  linTicks(lo,hiV,5).forEach(t=>{const yy=Y(t);
    if(yy==null||yy<m.t-1||yy>H-m.b+1)return;
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:yy,y2:yy,stroke:css('--grid'),'stroke-width':1}));
    const lb=el('text',{x:m.l-8,y:yy+4,'text-anchor':'end','font-size':11,fill:css('--muted')});
    lb.textContent=(100*t).toFixed(t<0.01?2:(t<0.1?1:0))+'%';svg.appendChild(lb);});
  for(let yv=YEARS[0];yv<=YEARS[YEARS.length-1];yv+=20){
    const lb=el('text',{x:x(yv),y:H-m.b+18,'text-anchor':'middle','font-size':11,fill:css('--muted')});
    lb.textContent=yv;svg.appendChild(lb);}
  svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:H-m.b,y2:H-m.b,stroke:css('--axis'),'stroke-width':1}));
  const xf=x(D.from);
  svg.appendChild(el('line',{x1:xf,x2:xf,y1:m.t,y2:H-m.b,stroke:css('--muted'),
    'stroke-width':1,'stroke-dasharray':'4 4'}));
  const a=el('text',{x:xf+5,y:m.t+12,'font-size':11,fill:css('--muted')});
  a.textContent=D.from;svg.appendChild(a);
  // draw ribbons first so the lines read on top of them
  if(bandOn())BAND.forEach(b=>{
    const lo=L[b.lo]||[], hi=L[b.hi]||[];
    let fwd='', back='';
    YEARS.forEach((yv,k)=>{ if(lo[k]==null||hi[k]==null) return;
      fwd += (fwd?'L':'M')+x(yv).toFixed(1)+' '+Y(hi[k]).toFixed(1)+' ';});
    for(let k=YEARS.length-1;k>=0;k--){ if(lo[k]==null||hi[k]==null) continue;
      back += 'L'+x(YEARS[k]).toFixed(1)+' '+Y(lo[k]).toFixed(1)+' ';}
    if(fwd)svg.appendChild(el('path',{d:fwd+back+'Z',fill:css(b.c),
      'fill-opacity':0.16,stroke:'none'}));});
  OBS.forEach(o=>{ if(!obsOn(o.k))return;
    (L[o.k]||[]).forEach(([yr,r,n,nb])=>{ if(r==null)return;
      const yy=Y(r); if(yy==null||yr<YEARS[0]||yr>YEARS[YEARS.length-1])return;
      const part=(nb!=null&&D.nBins!=null&&nb<D.nBins);
      svg.appendChild(el('circle',{cx:x(yr),cy:yy,
        r:Math.max(3,Math.min(7,Math.sqrt(n)/14)),fill:css(o.c),
        'fill-opacity':part?0:0.4,stroke:css(o.c),'stroke-width':1.2,
        'stroke-dasharray':part?'2 2':'none'}));});});
  vis().forEach(s=>{const arr=L[s.k]||[];let d='',pen=false;
    YEARS.forEach((yv,k)=>{const v=arr[k],yy=Y(v);
      if(v==null||yy==null){pen=false;return;}
      d+=(pen?'L':'M')+x(yv).toFixed(1)+' '+yy.toFixed(1)+' ';pen=true;});
    if(d)svg.appendChild(el('path',{d,fill:'none',stroke:css(s.c),'stroke-width':s.w,
      'stroke-linecap':'round','stroke-linejoin':'round',
      ...(s.d!=='none'?{'stroke-dasharray':s.d}:{})}));});
  const cross=el('g',{id:'cross',visibility:'hidden'});svg.appendChild(cross);
  cross.appendChild(el('line',{id:'cl',y1:m.t,y2:H-m.b,stroke:css('--muted'),'stroke-width':1}));
  const hit=el('rect',{x:m.l,y:m.t,width:Math.max(0,W-m.l-m.r),
    height:Math.max(0,H-m.t-m.b),fill:'transparent'});
  svg.appendChild(hit);
  hit.addEventListener('pointermove',onMove);
  hit.addEventListener('pointerleave',()=>{cross.setAttribute('visibility','hidden');tip.style.opacity=0;});
}
function linTicks(lo,hi,n){const raw=(hi-lo)/n||1e-9,mag=Math.pow(10,Math.floor(Math.log10(raw)));
  const st=[1,2,2.5,5,10].map(v=>v*mag).find(v=>v>=raw)||mag*10;const o=[];
  for(let v=Math.ceil(lo/st)*st;v<=hi;v+=st)o.push(v);return o;}
function onMove(ev){
  if(!geom)return;
  const r=svg.getBoundingClientRect(),sc=geom.W/r.width,px=(ev.clientX-r.left)*sc;
  let best=YEARS[0],bd=1e9;
  YEARS.forEach(y=>{const d=Math.abs(geom.x(y)-px);if(d<bd){bd=d;best=y;}});
  const i=YEARS.indexOf(best),L=geom.L;
  const cross=svg.querySelector('#cross');cross.setAttribute('visibility','visible');
  const cl=cross.querySelector('#cl');cl.setAttribute('x1',geom.x(best));cl.setAttribute('x2',geom.x(best));
  cross.querySelectorAll('circle').forEach(c=>c.remove());
  let rows='';
  vis().forEach(s=>{const v=(L[s.k]||[])[i];if(v==null)return;
    const yy=geom.Y(v);if(yy==null)return;
    cross.appendChild(el('circle',{cx:geom.x(best),cy:yy,r:4,fill:css(s.c),
      stroke:css('--surface'),'stroke-width':2}));
    rows+=`<tr><td><svg width="16" height="8"><line x1="0" y1="4" x2="16" y2="4"
      stroke="${css(s.c)}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''}/></svg> ${esc(s.l)}</td>
      <td class="v">${fmt(v)}</td></tr>`;});
  if(bandOn())BAND.forEach(b=>{const lo=(L[b.lo]||[])[i],hi=(L[b.hi]||[])[i];
    if(lo==null||hi==null)return;
    rows+=`<tr><td><svg width="16" height="8"><rect width="16" height="8"
      fill="${css(b.c)}" fill-opacity="0.16"/></svg> ${esc(b.l)}</td>
      <td class="v">${fmt(lo)}&ndash;${fmt(hi)}</td></tr>`;});
  OBS.forEach(o=>{ if(!obsOn(o.k))return;
    const h2=(L[o.k]||[]).find(([yr])=>yr===best); if(!h2)return;
    const part=(h2[3]!=null&&D.nBins!=null&&h2[3]<D.nBins);
    rows+=`<tr><td><svg width="16" height="8"><circle cx="8" cy="4" r="3.5"
      fill="${css(o.c)}" fill-opacity="${part?0:0.4}" stroke="${css(o.c)}"
      stroke-width="1.2" stroke-dasharray="${part?'2 2':'none'}"/></svg>
      ${esc(o.l)} <span style="color:var(--muted)">n=${h2[2].toLocaleString()}${
        part?`, ${h2[3]}/${D.nBins} bins`:''}</span></td>
      <td class="v">${fmt(h2[1])}</td></tr>`;});
  const flag=(L.over_s[i]?' &middot; <b>shifted: neonatal above</b>':'')
    +(L.over_o[i]?' &middot; <b>original: neonatal above</b>':'');
  tip.innerHTML=`<b>${esc(L.n)} &middot; ${best}</b>${flag}<table>${rows}</table>`;
  tip.style.opacity=1;
  const tw=tip.offsetWidth,th=tip.offsetHeight;
  let lft=ev.clientX+16; if(lft+tw>innerWidth-8)lft=ev.clientX-tw-16;
  tip.style.left=Math.max(8,lft)+'px';
  tip.style.top=Math.min(Math.max(8,ev.clientY-th/2),innerHeight-th-8)+'px';
}
function renderLegend(){
  const b=document.getElementById('legend');b.textContent='';
  SER.forEach(s=>{const x=document.createElement('button');
    x.className='lg';x.setAttribute('aria-pressed',String(!hidden.has(s.k)));
    x.innerHTML=`<svg width="22" height="10"><line x1="0" y1="5" x2="22" y2="5"
      stroke="${css(s.c)}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''} stroke-linecap="round"/></svg>
      <span>${esc(s.l)}</span>`;
    x.onclick=()=>{hidden.has(s.k)?hidden.delete(s.k):hidden.add(s.k);draw();};
    b.appendChild(x);});
  OBS.forEach(o=>{const z=document.createElement('button');
    z.className='lg'; z.setAttribute('aria-pressed',String(obsOn(o.k)));
    z.innerHTML=`<svg width="22" height="10"><circle cx="11" cy="5" r="4"
      fill="${css(o.c)}" fill-opacity="0.4" stroke="${css(o.c)}" stroke-width="1.2"/></svg>
      <span>${esc(o.l)}</span>`;
    z.onclick=()=>{hidden.has(o.k)?hidden.delete(o.k):hidden.add(o.k);draw();};
    b.appendChild(z);});
  const bb=document.createElement('button');
  bb.className='lg'; bb.setAttribute('aria-pressed',String(bandOn()));
  bb.innerHTML=`<svg width="22" height="10"><rect width="10" height="10"
    fill="${css('--u5')}" fill-opacity="0.16"/><rect x="12" width="10" height="10"
    fill="${css('--nn')}" fill-opacity="0.16"/></svg><span>95% of draws</span>`;
  bb.onclick=()=>{hidden.has('__band__')?hidden.delete('__band__'):hidden.add('__band__');draw();};
  b.appendChild(bb);
  [['shifted (full height)',10],['original (top strip)',4]].forEach(([lab,hh])=>{
    const t=document.createElement('span');t.className='lg';t.style.cursor='default';
    t.innerHTML=`<svg width="22" height="10"><rect width="22" height="${hh}" y="${(10-hh)/2}"
      fill="${css('--flag')}"/></svg><span>neonatal above: ${lab}</span>`;
    b.appendChild(t);});
}
function renderCards(){
  const U=D.summary;
    document.getElementById('cards').innerHTML=`
   <div class="card"><h3>Countries crossing</h3>
     <div class="big">${U.locs}</div><p>of ${U.totalLocs} modelled, after ${D.from}</p></div>
   <div class="card"><h3>Both models</h3><div class="big">${U.counts.both}</div>
     <p>shifted and original both invert</p></div>
   <div class="card"><h3>Original only</h3><div class="big">${U.counts.orig}</div>
     <p>inherited from inference</p></div>
   <div class="card"><h3>Shifted only</h3><div class="big">${U.counts.shift}</div>
     <p>introduced by the residual step</p></div>
   <div class="card"><h3>With survey data</h3><div class="big">${U.withObs}</div>
     <p>checkable against observations</p></div>`;
}
document.getElementById('nbins').textContent=D.nBins;
document.getElementById('theme').onclick=()=>{
  const now=document.documentElement.getAttribute('data-theme');
  const dark=now?now==='dark':matchMedia('(prefers-color-scheme: dark)').matches;
  document.documentElement.setAttribute('data-theme',dark?'light':'dark');
  draw();renderCards();};
document.getElementById('sect').onchange=e=>{
  sect=e.target.value; hi=0; renderList();
  // keep the chart inside the chosen section
  if(filtered.length&&!filtered.includes(cur)){cur=filtered[0];location.hash='loc='+cur;draw();}
};
addEventListener('resize',()=>{if(geom)drawChart(D.locs[cur]);});
const hashId=(location.hash.match(/loc=(\d+)/)||[])[1];
if(hashId&&D.locs[hashId])cur=hashId;
renderSect();renderList();renderCards();draw();
</script></body></html>
"""

_xp_doc = (
    XP_HTML.replace("__NNRV__", _xp_html.escape(XP_NN[1]))
    .replace("__CMRV__", _xp_html.escape(XP_CM[1]))
    .replace("__FROM__", str(XP_FROM))
    .replace("__SCEN__", XP_DRAW_SCENARIO)
    .replace(
        "__PAYLOAD__",
        _xp_json.dumps(XP_PAYLOAD, separators=(",", ":")).replace("</", "<\\/"),
    )
)
Path(XP_PATH).write_text(_xp_doc, encoding="utf-8")
print(f"Saved {XP_PATH} ({len(_xp_doc) / 1e6:.2f} MB)")

  observed rates: reusing xo_obs_* from the cell above
countries crossing after 2023: 186 of 204  (both 30, original only 91, shifted only 65); 52 have survey data
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/plots/neonatal_vs_under5_crossover_original.html (2.02 MB)
